In [1]:
from datetime import datetime, timezone
import json

from datasets import load_dataset
from huggingface_hub import HfApi

DATASET_REPO = "OpenClaw/clawhub-security-signals-live"

# The -live dataset keeps growing, so "main" isn't reproducible on its own.
# Resolve it to a concrete commit SHA right now, then load THAT commit —
# this is the freeze: whatever "main" does after this cell runs, re-running
# this notebook later will still pull the exact same snapshot.
api = HfApi()
dataset_info = api.dataset_info(DATASET_REPO, revision="main")
frozen_revision = dataset_info.sha
frozen_at = datetime.now(timezone.utc).isoformat()

dataset = load_dataset(
    DATASET_REPO,
    name="default",
    revision=frozen_revision,
    verification_mode="no_checks",  # carried over from the non-live dataset,
    # where the Hub's cached split sizes were stale relative to the actual
    # data files; keep it unless a run shows it's unnecessary here.
)

# -live ships a single split called "latest" — it avoids train/validation/test
# names because the corpus is refreshed rather than being a fixed benchmark.
latest = dataset["latest"]
print(latest[0]["skill_slug"], latest[0]["clawscan_verdict"])

# Provenance is tracked separately from the dataset itself (not as a row
# column) — one small file, written once per freeze.
provenance = {
    "source_dataset": DATASET_REPO,
    "source_dataset_revision": frozen_revision,
    "source_dataset_frozen_at": frozen_at,
}
with open("dataset_provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print(provenance)


jverdi/plaid clean
{'source_dataset': 'OpenClaw/clawhub-security-signals-live', 'source_dataset_revision': '323d4b4bd899884e6726b97d74158654281960e4', 'source_dataset_frozen_at': '2026-09-25T09:01:35.555373+00:00'}


In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)
print(f"dataset size: {latest.num_rows} rows x {len(latest.column_names)} columns")
latest.select(range(10)).to_pandas()


dataset size: 77873 rows x 25 columns


,id,skill_slug,skill_version,skill_md_content,skill_bundle_content,clawscan_verdict,clawscan_confidence,clawscan_model,clawscan_summary,static_status,static_finding_count,static_reason_codes,virustotal_status,virustotal_malicious_count,virustotal_suspicious_count,virustotal_harmless_count,virustotal_undetected_count,skillspector_status,skillspector_score,skillspector_severity,skillspector_issue_count,skillspector_issue_codes,skillspector_issue_categories,clawscan_context,split
0,cb4f0a1b6b483962458fa98032faeda93f7c2e6e13add9...,jverdi/plaid,0.0.3,---\nname: plaid\ndescription: plaid-cli a cli...,[],clean,medium,None,"The skill is coherent, but it routes Plaid fin...",clean,0,[],clean,0,0,0,66,clean,0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
1,3a8836f1f9e514380be3a63ac1ef33ff55f359aeb352d3...,steipete/apple-notes,1.0.0,---\nname: apple-notes\ndescription: Manage Ap...,[],clean,high,None,"This Apple Notes skill is purpose-aligned, but...",clean,0,[],clean,0,0,0,64,clean,9,LOW,2,"[SQP-2, SQP-2]",[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
2,d77e75c919d892fd58f7b18bfee7751a2a2a1729393d24...,steipete/apple-reminders,1.0.0,---\nname: apple-reminders\ndescription: Manag...,[],clean,high,None,This skill is a straightforward Apple Reminder...,clean,0,[],clean,0,0,0,63,clean,9,LOW,2,"[SQP-2, SQP-2]",[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
3,e8ec23687cf150ee061347ac5c76755aa1355ff57e435a...,steipete/bear-notes,1.0.0,"---\nname: bear-notes\ndescription: Create, se...",[],clean,high,None,This skill is a straightforward Bear notes hel...,clean,0,[],clean,0,0,0,66,clean,9,LOW,1,[SQP-2],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
4,d93023e42fc21a78891b1bddfc6a6a7b3c05dc5a96b8cd...,steipete/blogwatcher,1.0.0,---\nname: blogwatcher\ndescription: Monitor b...,[],clean,high,None,This skill is a straightforward wrapper for in...,clean,0,[],clean,0,0,0,66,clean,0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
5,bf9193ffaf93cf6e0d1249c87149e7af03b8c7f134bee9...,steipete/blucli,1.0.0,---\nname: blucli\ndescription: BluOS CLI (blu...,[],clean,high,None,"This skill is a small, purpose-aligned wrapper...",clean,0,[],clean,0,0,0,66,clean,0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
6,8037489a6d04baa393105633f2bffe58bf31439aac9636...,steipete/camsnap,1.0.0,---\nname: camsnap\ndescription: Capture frame...,[],clean,medium,None,"The skill is a small, coherent camera-capture ...",clean,0,[],suspicious,0,0,0,66,clean,0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
7,7e8320ccbd3885b996fe689f5985c0ca4d32cacbe864f1...,steipete/clawdhub,1.0.0,---\nname: clawdhub\ndescription: Use the Claw...,[],clean,high,None,"This skill is a coherent ClawdHub CLI helper, ...",clean,0,[],suspicious,0,0,0,64,clean,8,LOW,1,[SQP-2],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
8,8af7c46dc43cbbf5c2b709668e09336a7d933c7cfb3da5...,steipete/eightctl,1.0.0,---\nname: eightctl\ndescription: Control Eigh...,[],clean,high,None,"This skill is a small, disclosed wrapper for c...",clean,0,[],clean,0,0,0,63,clean,0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
9,f5047368657d4a314d1da874ba77fad087bd4aea84b3ac...,steipete/food-order,1.0.0,---\nname: food-order\ndescription: Reorder Fo...,[],clean,high,None,This Foodora ordering skill is coherent and in...,clean,0,[],suspicious,0,0,0,64,clean,8,LOW,1,[SQP-1],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest


In [3]:
df_all = latest.to_pandas()
df_all.head()


,id,skill_slug,skill_version,skill_md_content,skill_bundle_content,clawscan_verdict,clawscan_confidence,clawscan_model,clawscan_summary,static_status,static_finding_count,static_reason_codes,virustotal_status,virustotal_malicious_count,virustotal_suspicious_count,virustotal_harmless_count,virustotal_undetected_count,skillspector_status,skillspector_score,skillspector_severity,skillspector_issue_count,skillspector_issue_codes,skillspector_issue_categories,clawscan_context,split
0,cb4f0a1b6b483962458fa98032faeda93f7c2e6e13add9...,jverdi/plaid,0.0.3,---\nname: plaid\ndescription: plaid-cli a cli...,[],clean,medium,None,"The skill is coherent, but it routes Plaid fin...",clean,0,[],clean,0.0,0.0,0.0,66.0,clean,0.0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
1,3a8836f1f9e514380be3a63ac1ef33ff55f359aeb352d3...,steipete/apple-notes,1.0.0,---\nname: apple-notes\ndescription: Manage Ap...,[],clean,high,None,"This Apple Notes skill is purpose-aligned, but...",clean,0,[],clean,0.0,0.0,0.0,64.0,clean,9.0,LOW,2,"[SQP-2, SQP-2]",[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
2,d77e75c919d892fd58f7b18bfee7751a2a2a1729393d24...,steipete/apple-reminders,1.0.0,---\nname: apple-reminders\ndescription: Manag...,[],clean,high,None,This skill is a straightforward Apple Reminder...,clean,0,[],clean,0.0,0.0,0.0,63.0,clean,9.0,LOW,2,"[SQP-2, SQP-2]",[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
3,e8ec23687cf150ee061347ac5c76755aa1355ff57e435a...,steipete/bear-notes,1.0.0,"---\nname: bear-notes\ndescription: Create, se...",[],clean,high,None,This skill is a straightforward Bear notes hel...,clean,0,[],clean,0.0,0.0,0.0,66.0,clean,9.0,LOW,1,[SQP-2],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest
4,d93023e42fc21a78891b1bddfc6a6a7b3c05dc5a96b8cd...,steipete/blogwatcher,1.0.0,---\nname: blogwatcher\ndescription: Monitor b...,[],clean,high,None,This skill is a straightforward wrapper for in...,clean,0,[],clean,0.0,0.0,0.0,66.0,clean,0.0,LOW,0,[],[],"{'static': {'status': 'clean', 'verdict': 'cle...",latest


In [4]:
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter

# --- Stage 1: cheap pass over every skill to get its TOTAL version count ---
# One GET per skill (no version list, no per-version detail yet) — just enough
# to filter the whole registry down to the skills worth crawling in depth.
# Resumable: results are appended to a checkpoint file as they come in, and a
# re-run skips slugs already recorded there.
#
# The checkpoint is keyed by skill_slug, so it carries over from the earlier
# non-live dataset — switching to -live only adds the newly-listed skills to
# the todo list rather than re-crawling everything.

CHECKPOINT_PATH = Path("clawhub_version_counts.jsonl")
MAX_WORKERS = 32          # start conservative; ClawHub's real rate limit is unknown
MAX_RETRIES = 4
BASE_URL = "https://clawhub.ai/api/v1/skills/{slug}"

def load_checkpoint():
    done = {}
    if CHECKPOINT_PATH.exists():
        with CHECKPOINT_PATH.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                done[row["skill_slug"]] = row
    return done

def fetch_version_count(session, owner, slug):
    url = BASE_URL.format(slug=slug)
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, params={"ownerHandle": owner}, timeout=15)
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 200:
            data = resp.json()
            versions = (data.get("skill") or {}).get("stats", {}).get("versions")
            return {"status": "ok", "versions": versions}
        if resp.status_code == 429:
            wait = float(resp.headers.get("Retry-After", 2 * (attempt + 1)))
            time.sleep(wait)
            continue
        if resp.status_code == 404:
            return {"status": "not_found", "versions": None}
        if 500 <= resp.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
            continue
        return {"status": f"http_{resp.status_code}", "versions": None}
    return {"status": "failed_after_retries", "versions": None}

_write_lock = threading.Lock()

def process_one(session, skill_slug):
    owner, _, slug = skill_slug.partition("/")
    result = fetch_version_count(session, owner, slug)
    row = {"skill_slug": skill_slug, **result}
    with _write_lock:
        with CHECKPOINT_PATH.open("a") as f:
            f.write(json.dumps(row) + "\n")
    return row

done = load_checkpoint()
all_slugs = df_all["skill_slug"].unique().tolist()
todo = [s for s in all_slugs if s not in done]
print(f"{len(done)} already done, {len(todo)} remaining out of {len(all_slugs)} total.")

session = requests.Session()
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS * 2)
session.mount("https://", adapter)

processed = 0
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one, session, s): s for s in todo}
    for future in as_completed(futures):
        processed += 1
        if processed % 500 == 0 or processed == len(todo):
            elapsed = time.time() - start
            rate = processed / elapsed if elapsed > 0 else 0
            remaining = len(todo) - processed
            eta_min = remaining / rate / 60 if rate > 0 else float("inf")
            print(f"{processed}/{len(todo)} done ({rate:.1f}/s, ETA {eta_min:.1f} min)")

print("Stage 1 done.")


77873 already done, 0 remaining out of 77873 total.
Stage 1 done.


In [5]:
counts_df = pd.read_json(CHECKPOINT_PATH, lines=True, convert_dates=False)
print(counts_df["status"].value_counts())

merged = df_all.merge(counts_df[["skill_slug", "versions"]], on="skill_slug", how="left")
candidates = merged[merged["versions"] >= 2].copy()

print(f"{candidates['skill_slug'].nunique()} skills have 2+ versions on ClawHub "
      f"({len(candidates)} dataset rows).")
candidates[["skill_slug", "skill_version", "versions", "skillspector_score"]].head(20)


status
ok                      77098
not_found                 639
failed_after_retries       86
http_400                   42
http_401                    8
Name: count, dtype: int64
33527 skills have 2+ versions on ClawHub (33527 dataset rows).


,skill_slug,skill_version,versions,skillspector_score
0,jverdi/plaid,0.0.3,2.0,0.0
11,steipete/gifgrep,1.0.1,2.0,0.0
37,steipete/discord,1.0.1,2.0,9.0
40,shanelindsay/codex-orchestration,2.0.0,2.0,18.0
47,steipete/brave-search,1.0.1,2.0,74.0
52,steipete/nano-banana-pro,1.0.1,2.0,30.0
54,steipete/openai-image-gen,1.0.1,2.0,24.0
55,steipete/oracle,1.0.1,2.0,0.0
61,darinkishore/voice-transcribe,1.0.1,2.0,9.0
62,asleep123/caldav-calendar,1.0.1,2.0,0.0


In [6]:
# --- What does the version-count threshold actually cost? ---
# Uses the Stage 1 checkpoint already on disk. No network.
#
# A skill with N versions yields N-1 labelled instances: the first version has
# no predecessor, so there's no diff and no previous rating to compare against.
# That sets the hard floor at 2 versions, not 3. Anything above 2 is a judgement
# call, so it's worth seeing what each threshold buys and costs.

counts = pd.read_json(CHECKPOINT_PATH, lines=True, convert_dates=False)
ok_counts = counts[counts["status"] == "ok"].dropna(subset=["versions"]).copy()
ok_counts["versions"] = ok_counts["versions"].astype(int)

print("version-count distribution across successfully crawled skills:")
dist = ok_counts["versions"].value_counts().sort_index()
print(dist.head(12).to_string())
print(f"(max observed: {ok_counts['versions'].max()})\n")

total_pairs = (ok_counts["versions"] - 1).clip(lower=0).sum()
rows = []
for t in [2, 3, 4, 5, 10]:
    sel = ok_counts[ok_counts["versions"] >= t]
    pairs = int((sel["versions"] - 1).sum())
    rows.append({
        "threshold": f"{t}+ versions",
        "skills": len(sel),
        "pairs": pairs,
        "share_of_all_pairs": pairs / total_pairs if total_pairs else 0,
        "pairs_per_skill": pairs / len(sel) if len(sel) else 0,
        "extra_skills_vs_3plus": len(sel) - len(ok_counts[ok_counts["versions"] >= 3]),
    })

print(f"{total_pairs:,} labelled pairs exist in total (every skill with 2+ versions).\n")
print(pd.DataFrame(rows).to_string(index=False, formatters={
    "share_of_all_pairs": "{:.1%}".format,
    "pairs_per_skill": "{:.1f}".format,
}))

print("\nRead the 'share_of_all_pairs' column: that's how much of the available")
print("training data each threshold keeps. If 3+ already keeps most of it, the")
print("2-version skills are a lot of extra crawling for little extra signal —")
print("they contribute exactly one pair each. If it keeps much less, dropping to")
print("2+ is worth the extra hours, especially since the scanner-generation")
print("filter will cut the usable pairs further.")


version-count distribution across successfully crawled skills:
versions
1     43571
2     12254
3      6216
4      3814
5      2381
6      1720
7      1188
8       931
9       676
10      676
11      532
12      400
(max observed: 1294)

154,237 labelled pairs exist in total (every skill with 2+ versions).

   threshold  skills  pairs share_of_all_pairs pairs_per_skill  extra_skills_vs_3plus
 2+ versions   33527 154237             100.0%             4.6                  12254
 3+ versions   21273 141983              92.1%             6.7                      0
 4+ versions   15057 129551              84.0%             8.6                  -6216
 5+ versions   11243 118109              76.6%            10.5                 -10030
10+ versions    4347  80932              52.5%            18.6                 -16926

Read the 'share_of_all_pairs' column: that's how much of the available
training data each threshold keeps. If 3+ already keeps most of it, the
2-version skills are a lot of e

In [7]:
# --- Stage 2a: fetch the FULL version list for each of the candidates ---
# Reuses the session / MAX_WORKERS / MAX_RETRIES / _write_lock from Stage 1 above.
# If this errors with a NameError, re-run the Stage 1 cell first (kernel restarts
# lose in-memory variables even though the checkpoint files on disk are kept).

VERSIONS_CHECKPOINT_PATH = Path("clawhub_version_lists.jsonl")
VERSIONS_URL = "https://clawhub.ai/api/v1/skills/{slug}/versions"

def load_versions_checkpoint():
    done = {}
    if VERSIONS_CHECKPOINT_PATH.exists():
        with VERSIONS_CHECKPOINT_PATH.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                done[row["skill_slug"]] = row
    return done

def fetch_version_list(session, owner, slug):
    url = VERSIONS_URL.format(slug=slug)
    for attempt in range(MAX_RETRIES):
        try:
            # limit=200 covers virtually every skill (the CLI itself caps at 200).
            # hit_limit flags the rare skill that maxes it out, in case real
            # pagination (cursor param, unconfirmed field name) turns out to be needed.
            resp = session.get(url, params={"ownerHandle": owner, "limit": 200}, timeout=15)
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 200:
            data = resp.json()
            items = data.get("items") or []
            return {"status": "ok", "items": items, "hit_limit": len(items) >= 200}
        if resp.status_code == 429:
            wait = float(resp.headers.get("Retry-After", 2 * (attempt + 1)))
            time.sleep(wait)
            continue
        if resp.status_code == 404:
            return {"status": "not_found", "items": []}
        if 500 <= resp.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
            continue
        return {"status": f"http_{resp.status_code}", "items": []}
    return {"status": "failed_after_retries", "items": []}

def process_versions_one(session, skill_slug):
    owner, _, slug = skill_slug.partition("/")
    result = fetch_version_list(session, owner, slug)
    row = {"skill_slug": skill_slug, **result}
    with _write_lock:
        with VERSIONS_CHECKPOINT_PATH.open("a") as f:
            f.write(json.dumps(row) + "\n")
    return row

done_versions = load_versions_checkpoint()
candidate_slugs = candidates["skill_slug"].unique().tolist()
todo_versions = [s for s in candidate_slugs if s not in done_versions]
print(f"{len(done_versions)} already done, {len(todo_versions)} remaining out of {len(candidate_slugs)} candidates.")

processed = 0
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_versions_one, session, s): s for s in todo_versions}
    for future in as_completed(futures):
        processed += 1
        if processed % 250 == 0 or processed == len(todo_versions):
            elapsed = time.time() - start
            rate = processed / elapsed if elapsed > 0 else 0
            remaining = len(todo_versions) - processed
            eta_min = remaining / rate / 60 if rate > 0 else float("inf")
            print(f"{processed}/{len(todo_versions)} done ({rate:.1f}/s, ETA {eta_min:.1f} min)")

print("Stage 2a done.")


33527 already done, 0 remaining out of 33527 candidates.
Stage 2a done.


In [8]:
version_lists_df = pd.read_json(VERSIONS_CHECKPOINT_PATH, lines=True, convert_dates=False)
print(version_lists_df["status"].value_counts())

if "hit_limit" in version_lists_df and version_lists_df["hit_limit"].fillna(False).any():
    n_hit = int(version_lists_df["hit_limit"].fillna(False).sum())
    print(f"Warning: {n_hit} skill(s) returned exactly 200 versions — "
          f"real pagination may be needed for those to get their true full history.")

# Column names here are the final schema names, so nothing needs renaming later:
# owner_handle / changelog_text rather than owner / changelog.
flat_versions = []
for row in version_lists_df.itertuples():
    if row.status != "ok":
        continue
    owner, _, slug = row.skill_slug.partition("/")
    for item in (row.items or []):
        flat_versions.append({
            "skill_slug": row.skill_slug,
            "owner_handle": owner,
            "slug": slug,
            "version": item.get("version"),
            "created_at": item.get("createdAt"),   # epoch milliseconds
            "changelog_text": item.get("changelog"),
        })

versions_flat_df = pd.DataFrame(flat_versions)
print(f"{len(versions_flat_df)} total (skill, version) pairs across "
      f"{versions_flat_df['skill_slug'].nunique()} skills.")
versions_flat_df.head()


status
ok                      33492
failed_after_retries       32
not_found                   3
Name: count, dtype: int64
177339 total (skill, version) pairs across 33492 skills.


,skill_slug,owner_handle,slug,version,created_at,changelog_text
0,steipete/discord,steipete,discord,1.0.1,1767574303488,
1,steipete/discord,steipete,discord,1.0.0,1767545311074,
2,steipete/openai-image-gen,steipete,openai-image-gen,1.0.1,1767652000401,
3,steipete/openai-image-gen,steipete,openai-image-gen,1.0.0,1767545364210,
4,joargp/news-summary,joargp,news-summary,1.0.1,1767689990078,- Added detailed documentation of news sources...


In [9]:
# --- PROBE: what does the version endpoint actually return? ---
# Run this BEFORE the full Stage 2b crawl. It samples the oldest and newest
# version of a few skills, dumps the whole security object untruncated, and
# tabulates every field that appears: how often it is present, and how often it
# is actually non-null. That table decides which fields are worth extracting.
# Getting it wrong costs a re-crawl of ~98,000 calls; this costs ~16 calls.
#
# It also answers the "why nan vals?" question directly: fields that come back
# null on old versions will show up here with a low non_null count.

from collections import Counter

PROBE_SKILLS = 8

def flatten_keys(obj, prefix=""):
    """Every key path in a nested dict. Lists are summarised, not recursed."""
    out = {}
    if not isinstance(obj, dict):
        return out
    for k, v in obj.items():
        path = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flatten_keys(v, path))
            if not v:
                out[path] = "<empty dict>"
        elif isinstance(v, list):
            out[path] = f"<list of {len(v)}>" if v else None
        else:
            out[path] = v
    return out

def fetch_security(owner, slug, version):
    r = requests.get(
        f"https://clawhub.ai/api/v1/skills/{slug}/versions/{version}",
        params={"ownerHandle": owner}, timeout=20,
    )
    if r.status_code != 200:
        return None, r.status_code
    return (r.json().get("version") or {}).get("security"), 200

# Oldest and newest version of each of a few multi-version skills, so both
# scanner generations are represented.
ordered = versions_flat_df.sort_values(["skill_slug", "created_at"])
groups = ordered.groupby("skill_slug")
eligible = [s for s, grp in groups if len(grp) >= 3][:PROBE_SKILLS]

present, non_null, example = Counter(), Counter(), {}
records = []

for skill in eligible:
    grp = groups.get_group(skill)
    for label, row in [("oldest", grp.iloc[0]), ("newest", grp.iloc[-1])]:
        sec, code = fetch_security(row.owner_handle, row.slug, row.version)
        if sec is None:
            print(f"  skipped {skill} @ {row.version}: HTTP {code}")
            continue
        flat = flatten_keys(sec)
        for k, v in flat.items():
            present[k] += 1
            if v is not None:
                non_null[k] += 1
                example.setdefault(k, v)
        records.append((label, skill, row.version, sec, flat))

print(f"probed {len(records)} versions across {len(eligible)} skills\n")

# A path only gets a row of its own when its value is a SCALAR. A populated
# object is recorded as its child paths instead, so a parent path like
# "scanners.vt" appears ONLY on the versions where it was literally null.
# Reporting those as "always null" would be badly misleading - it would read as
# "vt is never populated" when it means the opposite. Separate the two kinds.
all_paths = set(present)
def is_container(p):
    return any(other.startswith(p + ".") for other in all_paths)

leaves = [p for p in sorted(all_paths) if not is_container(p)]
containers = [p for p in sorted(all_paths) if is_container(p)]

table = pd.DataFrame([
    {
        "field": k,
        "present": present[k],
        "non_null": non_null[k],
        "always_null": non_null[k] == 0,
        "example": str(example.get(k, ""))[:44],
    }
    for k in leaves
]).sort_values(["always_null", "field"])

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 46)
print("LEAF FIELDS (present = versions where this field carried a value):")
print(table.to_string(index=False))

never = list(table.loc[table["always_null"], "field"])
print("\nleaf fields never non-null in this sample (may still be populated "
      "elsewhere - do not assume they are useless):")
print("  " + (", ".join(never) or "none"))

if containers:
    print("\nNESTED OBJECTS - these were NULL on some versions and populated on")
    print("others. A null count here is NOT evidence the object is useless; when")
    print("it is populated its contents appear as the child fields above.")
    n_probed = len(records)
    con = pd.DataFrame([
        {
            "object": p,
            "null_on": present[p],
            "populated_on": n_probed - present[p],
            "child_fields": sum(1 for q in all_paths if q.startswith(p + ".")),
        }
        for p in containers
    ]).sort_values("object")
    print(con.to_string(index=False))

# Two full dumps for eyeballing: one old-pipeline, one new-pipeline.
old = next((r for r in records if not r[4].get("scanners.skillspector.score")), None)
new = next((r for r in records if r[4].get("scanners.skillspector.score")), None)

for label, rec in [("OLD PIPELINE (no skillspector)", old), ("NEW PIPELINE", new)]:
    print(f"\n{'=' * 70}\n{label}")
    if rec is None:
        print("  no example of this in the sample")
        continue
    _, skill, version, sec, _ = rec
    print(f"{skill} @ {version}\n{'=' * 70}")
    print(json.dumps(sec, indent=2, ensure_ascii=False))


probed 16 versions across 8 skills

LEAF FIELDS (present = versions where this field carried a value):
                                                             field  present  non_null  always_null                                        example
                                                         checkedAt       16        16        False                                  1774037840855
                                                     hasScanResult       16        16        False                                           True
                                                       hasWarnings       16        16        False                                           True
                                                             model       16         8        False                                     gpt-5-mini
                                  scanners.llm.agenticRiskFindings       16         2        False                                    <list of 3>
                     

In [11]:
# --- PROBE 2: is there a richer PACKAGES endpoint? ---
# From the clawhub CLI source:
#   - PackageFamilySchema is "skill"|"code-plugin"|"bundle-plugin"|"claw", so a
#     skill IS a package and the /api/v1/packages routes should apply.
#   - ApiV1PackageVersionResponseSchema (returned by
#     GET /api/v1/packages/{name}/versions/{version}) contains staticScan with a
#     findings[] array, and skillSpectorAnalysis with a FULL issues[] array
#     ({issueId, category, pattern, severity, confidence, file, startLine,
#      endLine, explanation, remediation, finding, codeSnippet}).
#   - It is called with getOptionalAuthToken(), not requireAuthToken - the same
#     pattern as /api/v1/download, which works anonymously.
#
# First attempt passed owner/slug in the path and got 404, but the BARE SLUG
# returned 409 AMBIGUOUS_SKILL_SLUG ("multiple skills with the slug ponytail").
# So the route does resolve skills; it just needs the same ?ownerHandle=
# disambiguator every other owner-scoped route in the CLI uses.

OWNER, SLUG = "dietrichgebert", "ponytail"
TEST_VERSIONS = ["4.9.0", "4.7.0"]      # newest, and an older one

ATTEMPTS = [
    ("bare slug + ownerHandle",        {"ownerHandle": OWNER}),
    ("bare slug + ownerHandle+family", {"ownerHandle": OWNER, "family": "skill"}),
    ("bare slug + owner",              {"owner": OWNER}),
    ("bare slug, no params",           {}),
]

def pkg_version(params, version):
    return requests.get(
        f"https://clawhub.ai/api/v1/packages/{SLUG}/versions/{version}",
        params=params, timeout=20,
    )

print("=== which parameter set resolves the package? ===")
working = None
for label, params in ATTEMPTS:
    r = pkg_version(params, TEST_VERSIONS[0])
    note = "OK" if r.status_code == 200 else r.text[:80]
    print(f"  {label:<34} -> HTTP {r.status_code}  {note}")
    if r.status_code == 200 and working is None:
        working = params

# Also confirm the package DETAIL route resolves, which tells us the family
# and the canonical name even if the version route needs something else.
print("\n=== package detail route ===")
d = requests.get(f"https://clawhub.ai/api/v1/packages/{SLUG}",
                 params={"ownerHandle": OWNER}, timeout=20)
print(f"  HTTP {d.status_code}")
if d.status_code == 200:
    pkg = (d.json().get("package") or {})
    print("  package keys:", sorted(pkg.keys())[:14])
    print(f"  name={pkg.get('name')!r}  family={pkg.get('family')!r}")
    print("  -> if 'name' differs from the slug, THAT is what the version route wants")
else:
    print(" ", d.text[:200])

if working is None:
    print("\nNo parameter set worked on the version route. Check the detail-route")
    print("output above for a canonical package name to use instead.")
else:
    print(f"\nworking params: {working}\n")
    for version in TEST_VERSIONS:
        r = pkg_version(working, version)
        print(f"{'=' * 66}\n{SLUG} @ {version}  (HTTP {r.status_code})\n{'=' * 66}")
        if r.status_code != 200:
            print(r.text[:200], "\n")
            continue
        v = r.json().get("version") or {}
        print("version keys:", sorted(v.keys()), "\n")

        ss = v.get("skillSpectorAnalysis")
        if ss:
            issues = ss.get("issues") or []
            print(f"skillSpectorAnalysis: score={ss.get('score')} "
                  f"severity={ss.get('severity')} issueCount={ss.get('issueCount')} "
                  f"scannerVersion={ss.get('scannerVersion')}")
            print(f"  issues[] has {len(issues)} entries")
            if issues:
                print("  first issue:\n   ", json.dumps(issues[0], indent=4)[:800])
        else:
            print("skillSpectorAnalysis: <null>")

        st = v.get("staticScan")
        if st:
            findings = st.get("findings") or []
            print(f"\nstaticScan: status={st.get('status')} "
                  f"engineVersion={st.get('engineVersion')} "
                  f"reasonCodes={st.get('reasonCodes')}")
            print(f"  findings[] has {len(findings)} entries")
            if findings:
                print("  first finding:", json.dumps(findings[0], indent=4)[:500])
        else:
            print("\nstaticScan: <null>")
        print()

    print("THE DECIDING QUESTION: are issues[] and findings[] populated on the")
    print("OLDER version too? If yes, per-issue detail exists across version")
    print("history and the LLM-dimensions workaround is no longer needed for the")
    print("label. If they appear only on the newest, it is the same re-scan")
    print("limitation in a different place and nothing changes.")


=== which parameter set resolves the package? ===
  bare slug + ownerHandle            -> HTTP 200  OK
  bare slug + ownerHandle+family     -> HTTP 200  OK
  bare slug + owner                  -> HTTP 200  OK
  bare slug, no params               -> HTTP 409  {"code":"AMBIGUOUS_SKILL_SLUG","message":"Found multiple skills with the slug \"

=== package detail route ===
  HTTP 200
  package keys: ['channel', 'compatibility', 'createdAt', 'displayName', 'family', 'isOfficial', 'latestVersion', 'name', 'ownerHandle', 'runtimeId', 'summary', 'tags', 'updatedAt', 'verification']
  name='ponytail'  family='skill'
  -> if 'name' differs from the slug, THAT is what the version route wants

working params: {'ownerHandle': 'dietrichgebert'}

ponytail @ 4.9.0  (HTTP 200)
version keys: ['artifact', 'changelog', 'compatibility', 'createdAt', 'distTags', 'files', 'verification', 'version'] 

skillSpectorAnalysis: <null>

staticScan: <null>

ponytail @ 4.7.0  (HTTP 200)
version keys: ['artifact', 'chan

In [13]:
# --- PROBE 3: does a LOGGED-IN request return more than an anonymous one? ---
# Probe 2 settled the anonymous case: /api/v1/packages/{slug}/versions/{version}
# answers HTTP 200 with a real skill body, but omits staticScan and
# skillSpectorAnalysis entirely. Two explanations survive that result, and they
# have opposite consequences for this dataset:
#
#   auth-gated   -> any signed-in account sees those fields. Attach a token to
#                   Stage 2b and static findings + per-issue SkillSpector detail
#                   become available for all ~98k versions.
#   owner-gated  -> only the skill's publisher sees them. Nothing changes; the
#                   five llm_dim_* ratings stay the only history-wide label.
#
# No token gets pasted into this notebook. The clawhub CLI writes its bearer
# token to disk on `clawhub login`, and this notebook runs on the same machine,
# so the cell reads it from there. The token is used and never printed.
#
# The control that actually separates the two explanations is at the bottom:
# the same authenticated calls against a skill THIS ACCOUNT PUBLISHED. If the
# fields appear there and nowhere else, it is ownership, not authentication.

import os
import sys

TEST_OWNER, TEST_SLUG = "dietrichgebert", "ponytail"
TEST_VERSIONS = ["4.9.0", "4.7.0"]          # newest, and an older one
PKG_URL = "https://clawhub.ai/api/v1/packages/{slug}/versions/{version}"
SKILL_URL = "https://clawhub.ai/api/v1/skills/{slug}/versions/{version}"

def clawhub_config_candidates():
    """Mirror getGlobalConfigPath() from the CLI's config.js, including the
    legacy 'clawdhub' directory it still falls back to."""
    override = (os.environ.get("CLAWHUB_CONFIG_PATH")
                or os.environ.get("CLAWDHUB_CONFIG_PATH"))
    if override:
        return [Path(override).expanduser()]
    home = Path.home()
    if sys.platform == "darwin":
        bases = [home / "Library" / "Application Support"]
    elif os.name == "nt":
        bases = [Path(os.environ["APPDATA"])] if os.environ.get("APPDATA") else []
    else:
        xdg = os.environ.get("XDG_CONFIG_HOME")
        bases = [Path(xdg)] if xdg else []
    bases.append(home / ".config")
    return [base / d / "config.json" for base in bases for d in ("clawhub", "clawdhub")]

TOKEN = None
for candidate in clawhub_config_candidates():
    if candidate.exists():
        cfg = json.loads(candidate.read_text(encoding="utf-8"))
        TOKEN = cfg.get("token")
        print(f"CLI config found at: {candidate}")
        break

if not TOKEN:
    print("No clawhub CLI token on disk. Looked in:")
    for candidate in clawhub_config_candidates():
        print("   ", candidate)
    print("\nRun `clawhub login` in a terminal, then re-run this cell.")
else:
    AUTH = {"Authorization": f"Bearer {TOKEN}"}
    print(f"token loaded ({len(TOKEN)} chars, not printed)")

    who = requests.get("https://clawhub.ai/api/v1/whoami", headers=AUTH, timeout=20)
    HANDLE = (who.json().get("user") or {}).get("handle") if who.status_code == 200 else None
    print(f"whoami -> HTTP {who.status_code}, handle={HANDLE!r}")
    if who.status_code != 200 or not HANDLE:
        print("Token is not being accepted. Re-run `clawhub login` before trusting"
              " anything below — an expired token would look exactly like"
              " 'auth unlocks nothing'.")

    # ---------------------------------------------------------------- helpers
    def key_paths(node, prefix=""):
        """Every dotted key path in a response, so an anon/auth diff catches
        fields this probe didn't think to look for."""
        paths = set()
        if isinstance(node, dict):
            for k, v in node.items():
                p = f"{prefix}.{k}" if prefix else k
                paths.add(p)
                paths |= key_paths(v, p)
        elif isinstance(node, list) and node:
            paths |= key_paths(node[0], f"{prefix}[]")
        return paths

    def get(url, params, headers):
        r = requests.get(url, params=params, headers=headers, timeout=20)
        return r.status_code, (r.json() if r.status_code == 200 else None)

    def describe(v, field, list_key):
        """absent / null / populated — the three states that matter here."""
        if field not in v:
            return "absent"
        obj = v.get(field)
        if obj is None:
            return "null"
        n = len(obj.get(list_key) or [])
        return f"present, {list_key}[]={n}"

    def compare(label, url, owner, slug, version):
        a_code, a = get(url.format(slug=slug, version=version),
                        {"ownerHandle": owner}, {})
        b_code, b = get(url.format(slug=slug, version=version),
                        {"ownerHandle": owner}, AUTH)
        print(f"\n--- {label}: {owner}/{slug} @ {version} ---")
        print(f"    anon HTTP {a_code}    auth HTTP {b_code}")
        if a_code != 200 and b_code != 200:
            print("    both failed; nothing to compare")
            return None
        av, bv = ((a or {}).get("version") or {}), ((b or {}).get("version") or {})
        print(f"    anon version keys: {sorted(av.keys())}")
        print(f"    auth version keys: {sorted(bv.keys())}")
        extra = key_paths(b) - key_paths(a)
        print(f"    key paths present ONLY when authenticated: "
              f"{sorted(extra) if extra else 'none'}")
        if label == "packages":
            print(f"    staticScan            anon={describe(av, 'staticScan', 'findings')}"
                  f"   auth={describe(bv, 'staticScan', 'findings')}")
            print(f"    skillSpectorAnalysis  anon={describe(av, 'skillSpectorAnalysis', 'issues')}"
                  f"   auth={describe(bv, 'skillSpectorAnalysis', 'issues')}")
        sec = (bv.get("security") or {})
        if sec:
            scanners = sec.get("scanners") or {}
            print(f"    security.scanners (auth): {sorted(scanners.keys())}")
            ss = scanners.get("skillspector")
            print(f"    scanners.skillspector keys (auth): "
                  f"{sorted(ss.keys()) if ss else '<null>'}")
        return extra

    # ------------------------------------------------- a skill she does NOT own
    print("\n" + "=" * 70)
    print("A. SKILL PUBLISHED BY SOMEONE ELSE  (the case that matters)")
    print("=" * 70)
    other_extra = set()
    for version in TEST_VERSIONS:
        for label, url in (("packages", PKG_URL), ("skills", SKILL_URL)):
            found = compare(label, url, TEST_OWNER, TEST_SLUG, version)
            if found:
                other_extra |= found

    # ----------------------------------------------------- a skill she DOES own
    print("\n" + "=" * 70)
    print("B. CONTROL: A SKILL THIS ACCOUNT PUBLISHED")
    print("=" * 70)
    own_extra = set()
    mine = []
    if HANDLE:
        s = requests.get("https://clawhub.ai/api/v1/search",
                         params={"q": HANDLE, "limit": 50}, headers=AUTH, timeout=20)
        if s.status_code == 200:
            for row in s.json().get("results", []):
                owner = row.get("ownerHandle") or ((row.get("owner") or {}).get("handle"))
                if owner == HANDLE and row.get("slug") and row.get("version"):
                    mine.append((owner, row["slug"], row["version"]))

    if not mine:
        print(f"No published skills found for {HANDLE!r}, so the control cannot run.")
        print("That leaves auth-gated and owner-gated indistinguishable ONLY IF")
        print("section A showed nothing — and in that case it does not matter,")
        print("because either way the fields are unreachable for other people's")
        print("skills, which is every skill in this dataset.")
    else:
        owner, slug, version = mine[0]
        print(f"testing own skill: {owner}/{slug} @ {version}  "
              f"({len(mine)} own skill(s) found)")
        for label, url in (("packages", PKG_URL), ("skills", SKILL_URL)):
            found = compare(label, url, owner, slug, version)
            if found:
                own_extra |= found

    # -------------------------------------------------------------- the verdict
    print("\n" + "=" * 70)
    print("VERDICT")
    print("=" * 70)
    gated = {"staticScan", "skillSpectorAnalysis"}
    unlocked_other = {p for p in other_extra if p.split(".")[-1].split("[")[0] in gated
                      or any(p.startswith(f"version.{g}") for g in gated)}
    unlocked_own = {p for p in own_extra if p.split(".")[-1].split("[")[0] in gated
                    or any(p.startswith(f"version.{g}") for g in gated)}

    if unlocked_other:
        print("AUTH-GATED. Authentication returned fields the anonymous call did not,")
        print("on a skill this account does not own:")
        for p in sorted(unlocked_other):
            print("   ", p)
        print("\nDo NOT start the 98k crawl as written. Stage 2b should be rewritten to")
        print("send the bearer token and to call the packages route as well as the")
        print("skills route, so static findings and per-issue SkillSpector detail are")
        print("captured while the crawl is running rather than needing a second pass.")
    elif unlocked_own:
        print("OWNER-GATED. The fields appear on this account's OWN skill but not on")
        print("someone else's, which is the ownership boundary, not an auth boundary.")
        print("Every skill in this dataset belongs to someone else, so this is closed:")
        print("static analysis and per-issue SkillSpector detail are not obtainable.")
        print("Proceed to Stage 2b as written, anonymously.")
    elif other_extra or own_extra:
        print("Authentication changed the response, but not in the fields at issue:")
        for p in sorted(other_extra | own_extra):
            print("   ", p)
        print("\nstaticScan and skillSpectorAnalysis stayed absent. Closed for this")
        print("purpose; proceed to Stage 2b as written, anonymously.")
    else:
        scope = ("on a skill this account owns AND one it does not"
                 if mine else
                 "on a skill this account does not own (no own skill to use as a control)")
        print("NOT EXPOSED AT ALL. The authenticated response has the same shape as")
        print(f"the anonymous one, {scope}.")
        print("staticScan and skillSpectorAnalysis are not served by the public API")
        print("in any form reachable from here.")
        print("\nThis closes the question. Stage 2b runs anonymously as written, and")
        print("the write-up should state that static-analysis findings are available")
        print("only at each skill's latest version, via the Hugging Face snapshot's")
        print("static_* columns, and cannot be differenced across versions.")


CLI config found at: C:\Users\theam\AppData\Roaming\clawhub\config.json
token loaded (47 chars, not printed)
whoami -> HTTP 200, handle='amm4ra'

A. SKILL PUBLISHED BY SOMEONE ELSE  (the case that matters)

--- packages: dietrichgebert/ponytail @ 4.9.0 ---
    anon HTTP 200    auth HTTP 200
    anon version keys: ['artifact', 'changelog', 'compatibility', 'createdAt', 'distTags', 'files', 'verification', 'version']
    auth version keys: ['artifact', 'changelog', 'compatibility', 'createdAt', 'distTags', 'files', 'verification', 'version']
    key paths present ONLY when authenticated: none
    staticScan            anon=absent   auth=absent
    skillSpectorAnalysis  anon=absent   auth=absent

--- skills: dietrichgebert/ponytail @ 4.9.0 ---
    anon HTTP 200    auth HTTP 200
    anon version keys: ['changelog', 'changelogSource', 'createdAt', 'files', 'license', 'security', 'version']
    auth version keys: ['changelog', 'changelogSource', 'createdAt', 'files', 'license', 'security', '

In [19]:
# --- Re-crawl truncated version lists with pagination (resumable) ---
import json, time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter

VERSIONS_JSONL = Path("clawhub_version_lists.jsonl")
PAGED_JSONL    = Path("clawhub_version_lists_paged.jsonl")   # this cell's own checkpoint
URL = "https://clawhub.ai/api/v1/skills/{slug}/versions"

session = requests.Session()
session.mount("https://", HTTPAdapter(pool_connections=4, pool_maxsize=8))

# preflight: fail fast if the host is unreachable rather than mid-crawl
try:
    session.get("https://clawhub.ai/api/v1/skills", params={"limit": 1}, timeout=20)
    print("clawhub.ai reachable")
except requests.RequestException as exc:
    raise SystemExit(f"cannot reach clawhub.ai: {exc}")

def get_page(slug, params):
    for attempt in range(6):
        try:
            r = session.get(URL.format(slug=slug), params=params, timeout=30)
        except requests.RequestException:
            time.sleep(min(2 ** attempt, 30))     # covers refused / reset / DNS
            continue
        if r.status_code == 200:
            return r.json(), "ok"
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 2 * (attempt + 1))))
        elif 500 <= r.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
        else:
            return None, f"http_{r.status_code}"
    return None, "failed_after_retries"

def fetch_all(owner, slug):
    items, cursor, pages = [], None, 0
    while True:
        params = {"ownerHandle": owner, "limit": 50}
        if cursor:
            params["cursor"] = cursor
        d, status = get_page(slug, params)
        if status != "ok":
            return items, status
        items += d.get("items", [])
        pages += 1
        cursor = d.get("nextCursor")
        if not cursor or pages >= 200:
            return items, "ok"
        time.sleep(0.1)

done = {}
if PAGED_JSONL.exists():
    for line in PAGED_JSONL.read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            done[r["skill_slug"]] = r

lists_df = pd.read_json(VERSIONS_JSONL, lines=True, convert_dates=False)
ok = lists_df[lists_df["status"] == "ok"].copy()
ok["n"] = ok["items"].map(len)
ceiling = ok.loc[ok["n"] >= 50, "skill_slug"].tolist()
todo = [s for s in ceiling if s not in done or done[s]["status"] != "ok"]
print(f"{len(ceiling)} truncated skills; {len(done)} already paged; {len(todo)} to go\n")

for i, skill_slug in enumerate(todo, 1):
    owner, _, slug = skill_slug.partition("/")
    items, status = fetch_all(owner, slug)
    row = {"skill_slug": skill_slug, "status": status, "items": items, "hit_limit": False}
    with PAGED_JSONL.open("a") as f:
        f.write(json.dumps(row) + "\n")
    done[skill_slug] = row
    if i % 10 == 0 or i == len(todo):
        print(f"  {i}/{len(todo)}  last: {skill_slug} -> {status}, {len(items)} versions")

statuses = pd.Series([done[s]["status"] for s in ceiling if s in done]).value_counts()
print(f"\nstatuses: {statuses.to_dict()}")
missing = [s for s in ceiling if s not in done or done[s]["status"] != "ok"]
print(f"still incomplete: {len(missing)}")

clawhub.ai reachable
152 truncated skills; 0 already paged; 152 to go

  10/152  last: onlyloveher/shield-cn -> ok, 150 versions
  20/152  last: shenmeng/shenmeng-income-lab -> ok, 50 versions
  30/152  last: mouxangithub/unified-memory -> ok, 50 versions
  40/152  last: parkertoddbrooks/wip-repos -> ok, 50 versions
  50/152  last: sipingme/wechat-md-publisher-skill -> ok, 50 versions
  60/152  last: liyafeichina/write-a-skill -> ok, 50 versions
  70/152  last: cjlrestlong-ai/hk-stock-morning-report -> ok, 50 versions
  80/152  last: batthis/amber-phone-agent -> ok, 50 versions
  90/152  last: polyparlay/hyperliquid-place -> ok, 50 versions
  100/152  last: loonghao/dcc-mcp-skills-creator -> ok, 50 versions
  110/152  last: simmer/polymarket-weather-trader -> ok, 50 versions
  120/152  last: xor777/magicpay -> ok, 50 versions
  130/152  last: thesentitrader/sentisense -> ok, 50 versions
  140/152  last: zw008/vmware-monitor -> ok, 50 versions
  150/152  last: mnetfairy/ai-insurance-adv

In [41]:
import json, time, requests
import pandas as pd

counts = pd.read_json("clawhub_version_counts.jsonl", lines=True, convert_dates=False)
expected = counts[counts["status"] == "ok"].set_index("skill_slug")["versions"]
paged = [json.loads(l) for l in open("clawhub_version_lists_paged.jsonl") if l.strip()]

rows = []
for r in paged:
    slug = r["skill_slug"]
    rows.append({"skill_slug": slug,
                 "stage1_count": expected.get(slug),
                 "paged_count": len(r["items"])})
d = pd.DataFrame(rows)
d["complete"] = d["paged_count"] >= d["stage1_count"]

print(f"{len(d)} skills\n")
print("versions retrieved by pagination:")
print(d["paged_count"].value_counts().sort_index().to_string())
print(f"\ncomplete (paged >= stage1): {int(d['complete'].sum())} / {len(d)}")
print(f"\nskills where stage1 says >150: {int((d['stage1_count'] > 150).sum())}")
print(d[d["stage1_count"] > 150].sort_values("stage1_count", ascending=False)
       .head(20).to_string(index=False))

152 skills

versions retrieved by pagination:
paged_count
50     149
100      2
150      1

complete (paged >= stage1): 5 / 152

skills where stage1 says >150: 22
                             skill_slug  stage1_count  paged_count  complete
                  onlyloveher/shield-cn        1294.0          150     False
  shenmeng/shenmeng-whale-alert-monitor         656.0          100     False
shenmeng/shenmeng-odds-movement-monitor         615.0          100     False
             liyafeichina/write-a-skill         473.0           50     False
                         arberx/canonry         457.0           50     False
     liuyanfeng1234/v19-trust-manifesto         280.0           50     False
                  wazionapps/nexo-brain         245.0           50     False
        mnetfairy/ai-era-career-planner         242.0           50     False
         mnetfairy/career-planner-china         237.0           50     False
         mnetfairy/ai-insurance-advisor         235.0           50 

In [20]:
# --- REHYDRATE: everything Stage 2b needs, after a kernel restart ---
# Stage 2b depends on names defined in the Stage 1 and Stage 2a cells
# (session, MAX_WORKERS, MAX_RETRIES, _write_lock, versions_flat_df). A kernel
# restart loses those, but every crawl already finished is on DISK, so none of
# it needs re-running: this cell rebuilds the names from the checkpoint files
# alone. No Hugging Face download, no Stage 1, no Stage 2a re-crawl.
#
# Run this, then go straight to Stage 2b.

import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter

MAX_WORKERS = 48
MAX_RETRIES = 4
_write_lock = threading.Lock()

session = requests.Session()
session.mount("https://", HTTPAdapter(pool_connections=MAX_WORKERS,
                                      pool_maxsize=MAX_WORKERS * 2))

VERSIONS_CHECKPOINT_PATH = Path("clawhub_version_lists.jsonl")
if not VERSIONS_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"{VERSIONS_CHECKPOINT_PATH} is missing — Stage 2a has not run in this "
        f"working directory. Run the notebook from the top instead."
    )

version_lists_df = pd.read_json(VERSIONS_CHECKPOINT_PATH, lines=True, convert_dates=False)

flat_versions = []
for row in version_lists_df.itertuples():
    if row.status != "ok":
        continue
    owner, _, slug = row.skill_slug.partition("/")
    for item in (row.items or []):
        flat_versions.append({
            "skill_slug": row.skill_slug,
            "owner_handle": owner,
            "slug": slug,
            "version": item.get("version"),
            "created_at": item.get("createdAt"),
            "changelog_text": item.get("changelog"),
            "changelog_source": item.get("changelogSource"),
        })

versions_flat_df = pd.DataFrame(flat_versions)
print(f"rebuilt versions_flat_df: {len(versions_flat_df):,} (skill, version) pairs "
      f"across {versions_flat_df['skill_slug'].nunique():,} skills")
print(f"session + MAX_WORKERS={MAX_WORKERS} + _write_lock ready. Stage 2b can run.")


rebuilt versions_flat_df: 177,339 (skill, version) pairs across 33,492 skills
session + MAX_WORKERS=48 + _write_lock ready. Stage 2b can run.


In [21]:
# --- Stage 2b: full security record for every (skill, version) pair ---
# One call per version. Same resumable checkpoint pattern as before.
#
# WHY THIS PULLS MORE THAN SKILLSPECTOR: 
# ClawHub added SkillSpector and VirusTotal to its scan pipeline partway through
# 2026, and only ever re-scans a skill's CURRENTLY LIVE version with the newer
# pipeline. Older versions keep whatever their original scan recorded, so
# scanners.skillspector and scanners.vt come back as literal null for most of
# them. scanners.llm, though, is populated for essentially every version — with
# a verdict, a confidence, and five consistently-named dimensions each rated
# ok / note / concern plus prose detail. That's the per-version security outcome
# the JIT label needs, so capture all three scanners and let the analysis choose.
#
# New checkpoint filename, so the earlier skillspector-only crawl stays intact.
# This file gets large: the prose runs a few KB per version.

VERSION_DETAIL_CHECKPOINT_PATH = Path("clawhub_version_security.jsonl")
VERSION_DETAIL_URL = "https://clawhub.ai/api/v1/skills/{slug}/versions/{version}"

# The five dimensions the LLM scanner reports. Anything unexpected is still kept
# in llm_dimensions_json, so a new dimension name can't be silently dropped.
LLM_DIMENSIONS = [
    "purpose_capability",
    "instruction_scope",
    "install_mechanism",
    "environment_proportionality",
    "persistence_privilege",
]

def load_detail_checkpoint():
    done = set()
    if VERSION_DETAIL_CHECKPOINT_PATH.exists():
        with VERSION_DETAIL_CHECKPOINT_PATH.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                done.add((row["skill_slug"], row["version"]))
    return done

def parse_security(data):
    """Pull out everything the response offers.

    Field choices are based on the probe cell's output, not guesswork:
      - vt.analysis is a PROSE security assessment on ~70% of versions, not
        engine counts. Easy to miss and worth keeping.
      - vt.source takes two values: "palm" (model-based, carries prose
        analysis + verdict) and "engines" (antivirus, analysis/verdict null).
        These MOSTLY track the pipeline change but do NOT map onto it cleanly:
        in the 16-version probe, vt.analysis was non-null on 9 versions while
        only 8 were old-pipeline, so at least one new-pipeline version also
        carries prose. Read vt_source per row rather than inferring it from
        whether SkillSpector is present.
      - llm.riskSummary is a NESTED object (three named risk areas, each with
        status/highestSeverity/summary), and agenticRiskFindings is a LIST.
        Both are stored as JSON so nothing is lost; flatten them locally later
        if useful - that needs no re-crawl.
    virustotalUrl is deliberately skipped: it is just
    https://www.virustotal.com/gui/file/<sha256hash>, so it carries nothing
    the hash does not already give us."""
    v = data.get("version") or {}
    sec = v.get("security") or {}
    scanners = sec.get("scanners") or {}
    ss = scanners.get("skillspector") or {}
    llm = scanners.get("llm") or {}
    vt = scanners.get("vt") or {}

    dims = {}
    for entry in (llm.get("dimensions") or []):
        if entry.get("name"):
            dims[entry["name"]] = entry.get("rating")

    def as_json(value):
        return json.dumps(value, ensure_ascii=False) if value else None

    out = {
        "crawled_detail_status": "ok",
        # --- overall verdict for this version
        "crawled_security_status": sec.get("status"),
        "crawled_security_checked_at": sec.get("checkedAt"),
        "crawled_has_scan_result": sec.get("hasScanResult"),
        "crawled_has_warnings": sec.get("hasWarnings"),
        "crawled_security_model": sec.get("model"),
        # content hash - equal hashes mean byte-identical bundles
        "crawled_sha256": sec.get("sha256hash"),
        # --- SkillSpector: only on versions scanned by the newer pipeline
        "crawled_skillspector_status": ss.get("status"),
        "crawled_skillspector_normalized_status": ss.get("normalizedStatus"),
        "crawled_skillspector_score": ss.get("score"),
        "crawled_skillspector_severity": ss.get("severity"),
        "crawled_skillspector_recommendation": ss.get("recommendatiowhn"),
        "crawled_skillspector_issue_count": ss.get("issueCount"),
        "crawled_skillspector_checked_at": ss.get("checkedAt"),
        # --- LLM scanner (ClawScan's own verdict): on essentially every version
        "llm_status": llm.get("status"),
        "llm_verdict": llm.get("verdict"),
        "llm_normalized_status": llm.get("normalizedStatus"),
        "llm_confidence": llm.get("confidence"),
        "llm_summary": llm.get("summary"),
        "llm_guidance": llm.get("guidance"),
        "llm_findings": llm.get("findings"),
        "llm_model": llm.get("model"),
        "llm_checked_at": llm.get("checkedAt"),
        "llm_dimensions_json": as_json(llm.get("dimensions")),
        "llm_risk_summary_json": as_json(llm.get("riskSummary")),
        "llm_agentic_risk_findings_json": as_json(llm.get("agenticRiskFindings")),
        # --- VirusTotal (meaning depends on vt_source, see docstring)
        "vt_status": vt.get("status"),
        "vt_verdict": vt.get("verdict"),
        "vt_normalized_status": vt.get("normalizedStatus"),
        "vt_source": vt.get("source"),
        "vt_analysis": vt.get("analysis"),
        "vt_checked_at": vt.get("checkedAt"),
    }
    for name in LLM_DIMENSIONS:
        out[f"llm_dim_{name}"] = dims.get(name)
    return out

def fetch_version_detail(session, owner, slug, version):
    url = VERSION_DETAIL_URL.format(slug=slug, version=version)
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, params={"ownerHandle": owner}, timeout=15)
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 200:
            return parse_security(resp.json())
        if resp.status_code == 429:
            wait = float(resp.headers.get("Retry-After", 2 * (attempt + 1)))
            time.sleep(wait)
            continue
        if resp.status_code == 404:
            return {"crawled_detail_status": "not_found"}
        if 500 <= resp.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
            continue
        return {"crawled_detail_status": f"http_{resp.status_code}"}
    return {"crawled_detail_status": "failed_after_retries"}

def process_detail_one(session, skill_slug, owner, slug, version):
    result = fetch_version_detail(session, owner, slug, version)
    row = {"skill_slug": skill_slug, "version": version, **result}
    with _write_lock:
        with VERSION_DETAIL_CHECKPOINT_PATH.open("a") as f:
            f.write(json.dumps(row) + "\n")
    return row

done_details = load_detail_checkpoint()
todo_details = [
    row for row in versions_flat_df.itertuples()
    if (row.skill_slug, row.version) not in done_details
]
print(f"{len(done_details)} already done, {len(todo_details)} remaining out of {len(versions_flat_df)} total versions.")

processed = 0
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_detail_one, session, row.skill_slug, row.owner_handle, row.slug, row.version): row
        for row in todo_details
    }
    for future in as_completed(futures):
        processed += 1
        if processed % 1000 == 0 or processed == len(todo_details):
            elapsed = time.time() - start
            rate = processed / elapsed if elapsed > 0 else 0
            remaining = len(todo_details) - processed
            eta_min = remaining / rate / 60 if rate > 0 else float("inf")
            print(f"{processed}/{len(todo_details)} done ({rate:.1f}/s, ETA {eta_min:.1f} min)")

print("Stage 2b done.")


177339 already done, 0 remaining out of 177339 total versions.
Stage 2b done.


In [22]:
# --- Merge Stage 2a (version list + changelog) with Stage 2b (security) ---
# This is an intermediate table, not the final dataset — Stage 3 below adds the
# diffs and per-version content, and the last cell assembles everything.

details_df = pd.read_json(VERSION_DETAIL_CHECKPOINT_PATH, lines=True, convert_dates=False)
print(details_df["crawled_detail_status"].value_counts())

version_history_df = versions_flat_df.merge(details_df, on=["skill_slug", "version"], how="left")
version_history_df = version_history_df.sort_values(["skill_slug", "created_at"]).reset_index(drop=True)

print(f"Version history so far: {len(version_history_df)} rows across "
      f"{version_history_df['skill_slug'].nunique()} skills.")

version_history_df.to_parquet("clawhub_version_history.parquet", index=False)
version_history_df.head(20)


crawled_detail_status
ok                      176659
failed_after_retries       487
not_found                  193
Name: count, dtype: int64
Version history so far: 177339 rows across 33492 skills.


,skill_slug,owner_handle,slug,version,created_at,changelog_text,changelog_source,crawled_detail_status,crawled_security_status,crawled_security_checked_at,...,vt_verdict,vt_normalized_status,vt_source,vt_analysis,vt_checked_at,llm_dim_purpose_capability,llm_dim_instruction_scope,llm_dim_install_mechanism,llm_dim_environment_proportionality,llm_dim_persistence_privilege
0,000wonderclaw/clawcv,000wonderclaw,clawcv,1.0.0,1774018324946,Initial release of ClawCV resume optimization ...,user,ok,suspicious,1.774038e+12,...,benign,clean,palm,Type: OpenClaw Skill\nName: clawcv\nVersion: 1...,1.774038e+12,concern,note,note,concern,ok
1,000wonderclaw/clawcv,000wonderclaw,clawcv,1.0.2,1774326884515,- Added English README file (README_EN.md) for...,user,ok,clean,1.774327e+12,...,benign,clean,palm,Type: OpenClaw Skill\nName: clawcv\nVersion: 1...,1.774327e+12,ok,ok,note,ok,ok
2,000wonderclaw/clawcv,000wonderclaw,clawcv,1.1.0,1775556109053,ClawCV 1.1.0 introduces campus recruitment job...,user,ok,suspicious,1.789420e+12,...,None,clean,engines,None,1.780086e+12,note,concern,concern,concern,concern
3,00xmorty/conatus,00xmorty,conatus,1.0.0,1772063119292,The philosophical layer for AI agents. Spinoza...,user,ok,clean,1.772063e+12,...,None,None,None,None,NaN,ok,note,ok,note,ok
4,00xmorty/conatus,00xmorty,conatus,1.0.1,1772063689744,Fix: republish to registry,user,ok,suspicious,1.789086e+12,...,benign,clean,palm,Type: OpenClaw Skill\nName: conatus\nVersion: ...,1.779167e+12,note,concern,ok,concern,concern
5,05u/jiuwu-agent,05u,jiuwu-agent,1.0.0,1774407899698,- Introduced jiuwu-agent skill for intelligent...,user,ok,suspicious,1.774410e+12,...,benign,clean,palm,Type: OpenClaw Skill\nName: jiuwu-agent\nVersi...,1.774410e+12,concern,note,ok,concern,ok
6,05u/jiuwu-agent,05u,jiuwu-agent,1.0.1,1775091906589,- Added scripts/requirements.txt to manage Pyt...,user,ok,suspicious,1.789435e+12,...,None,clean,engines,None,1.780090e+12,concern,concern,note,concern,ok
7,05u/jiuwu-message,05u,jiuwu-message,1.0.0,1773882295554,- Initial release of jiuwu-message skill.\n- E...,user,ok,suspicious,1.775029e+12,...,benign,clean,palm,Type: OpenClaw Skill\nName: jiuwu-message\nVer...,1.775029e+12,note,concern,ok,concern,ok
8,05u/jiuwu-message,05u,jiuwu-message,1.0.1,1775091842974,- Added initial scripts and requirements files...,user,ok,suspicious,1.789195e+12,...,None,clean,engines,None,1.780086e+12,note,concern,ok,concern,note
9,0731coderlee-sudo/wechat-publisher,0731coderlee-sudo,wechat-publisher,1.0.0,1770288770604,一键发布 Markdown 到微信公众号草稿箱,user,ok,pending,NaN,...,None,None,None,None,NaN,None,None,None,None,None


In [5]:
# --- Check the download endpoint before committing to a multi-hour crawl ---
# From the clawhub CLI's own source (http.js, downloadZipRequest): it issues
# GET /api/v1/download?slug=&ownerHandle=&version= and only attaches an
# Authorization header IF a token happens to exist. So it should work
# unauthenticated, on any public skill, for any specific historical version —
# including skills we don't own. Verify that here on ponytail 4.9.0 before
# trusting it at scale.

import io
import zipfile

check = requests.get(
    "https://clawhub.ai/api/v1/download",
    params={"slug": "ponytail", "ownerHandle": "dietrichgebert", "version": "4.9.0"},
    timeout=30,
)
print(check.status_code, check.headers.get("content-type"), len(check.content), "bytes")

with zipfile.ZipFile(io.BytesIO(check.content)) as zf:
    names = zf.namelist()
    print("files in bundle:", names)
    skill_md = next((n for n in names if n.upper().endswith("SKILL.MD")), None)
    if skill_md:
        text = zf.read(skill_md).decode("utf-8")
        print(f"\n{skill_md} — {len(text)} chars. First 400:\n")
        print(text[:400])


200 application/zip 4666 bytes
files in bundle: ['skill-card.md', 'SKILL.md', '_meta.json']

SKILL.md — 5936 chars. First 400:

---
name: ponytail
description: "Lazy senior dev mode for any coding task (write, refactor, fix, review): YAGNI, stdlib first, no unrequested abstractions. Not for non-coding requests."
homepage: https://github.com/DietrichGebert/ponytail
license: MIT
---

# Ponytail

You are a lazy senior developer. Lazy means efficient, not careless. You have
seen every over-engineered codebase and been paged at


In [23]:
# --- Stage 3a: download each version's actual files, so they can be diffed ---
# One zip per version, so this is the heaviest stage. Content is cached to disk
# (gzipped JSON, one file per version) and a light index tracks what's done, so
# the crawl is resumable and the diffing step below needs no network at all.
#
# Still depends on Stage 1 for the shared session / _write_lock / MAX_WORKERS /
# MAX_RETRIES — re-run that cell first if this raises a NameError for those.

import gzip
import io
import json
import re
import time
import zipfile
from pathlib import Path

CONTENT_DIR = Path("clawhub_content")
CONTENT_INDEX_PATH = Path("clawhub_content_index.jsonl")
DOWNLOAD_URL = "https://clawhub.ai/api/v1/download"

MAX_FILE_BYTES = 1_000_000     # skip any single file larger than this
MAX_TOTAL_BYTES = 5_000_000    # cap per version, so one huge bundle can't eat the disk

# Start small. This stage is several hours at full scale, so run it once with a
# small number, check the output, THEN set this to None for the real crawl.
SMOKE_TEST_SKILLS = 50

def safe_name(value):
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(value))

def content_path(skill_slug, version):
    owner, _, slug = skill_slug.partition("/")
    return CONTENT_DIR / safe_name(owner) / f"{safe_name(slug)}__{safe_name(version)}.json.gz"

def extract_text_files(zip_bytes):
    """Pull every UTF-8-decodable file out of the bundle. Binaries are skipped —
    they can't be diffed as text and aren't what the card sort is about."""
    files = {}
    total = 0
    truncated = False
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for info in sorted(zf.infolist(), key=lambda i: i.filename):
            if info.is_dir() or info.file_size > MAX_FILE_BYTES:
                continue
            try:
                text = zf.read(info).decode("utf-8")
            except (UnicodeDecodeError, RuntimeError):
                continue
            if total + len(text) > MAX_TOTAL_BYTES:
                truncated = True
                break
            files[info.filename] = text
            total += len(text)
    return files, total, truncated

def fetch_and_store_content(session, skill_slug, owner, slug, version):
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(
                DOWNLOAD_URL,
                params={"slug": slug, "ownerHandle": owner, "version": version},
                timeout=60,
            )
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 200:
            try:
                files, total, truncated = extract_text_files(resp.content)
            except zipfile.BadZipFile:
                return {"status": "bad_zip"}
            path = content_path(skill_slug, version)
            path.parent.mkdir(parents=True, exist_ok=True)
            with gzip.open(path, "wt", encoding="utf-8") as f:
                json.dump(files, f)
            return {"status": "ok", "n_files": len(files),
                    "n_chars": total, "truncated": truncated}
        if resp.status_code == 429:
            time.sleep(float(resp.headers.get("Retry-After", 2 * (attempt + 1))))
            continue
        if resp.status_code == 404:
            return {"status": "not_found"}
        if 500 <= resp.status_code < 600:
            time.sleep(1.5 * (attempt + 1))
            continue
        return {"status": f"http_{resp.status_code}"}
    return {"status": "failed_after_retries"}

def process_content_one(session, skill_slug, owner, slug, version):
    result = fetch_and_store_content(session, skill_slug, owner, slug, version)
    row = {"skill_slug": skill_slug, "version": version, **result}
    with _write_lock:
        with CONTENT_INDEX_PATH.open("a") as f:
            f.write(json.dumps(row) + "\n")
    return row

def load_content_checkpoint():
    done = set()
    if CONTENT_INDEX_PATH.exists():
        with CONTENT_INDEX_PATH.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                done.add((row["skill_slug"], row["version"]))
    return done

target_versions = versions_flat_df
if SMOKE_TEST_SKILLS is not None:
    keep = versions_flat_df["skill_slug"].drop_duplicates().head(SMOKE_TEST_SKILLS)
    target_versions = versions_flat_df[versions_flat_df["skill_slug"].isin(keep)]
    print(f"*** SMOKE TEST: {len(keep)} skills / {len(target_versions)} versions only. "
          f"Set SMOKE_TEST_SKILLS = None for the full crawl. ***")

done_content = load_content_checkpoint()
todo_content = [
    row for row in target_versions.itertuples()
    if (row.skill_slug, row.version) not in done_content
]
print(f"{len(done_content)} already downloaded, {len(todo_content)} remaining.")

processed = 0
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_content_one, session, row.skill_slug,
                        row.owner_handle, row.slug, row.version): row
        for row in todo_content
    }
    for future in as_completed(futures):
        processed += 1
        if processed % 250 == 0 or processed == len(todo_content):
            elapsed = time.time() - start
            rate = processed / elapsed if elapsed > 0 else 0
            remaining = len(todo_content) - processed
            eta_min = remaining / rate / 60 if rate > 0 else float("inf")
            print(f"{processed}/{len(todo_content)} done ({rate:.1f}/s, ETA {eta_min:.1f} min)")

content_index_df = pd.read_json(CONTENT_INDEX_PATH, lines=True, convert_dates=False)
print("\nStage 3a done.")
print(content_index_df["status"].value_counts())


*** SMOKE TEST: 50 skills / 140 versions only. Set SMOKE_TEST_SKILLS = None for the full crawl. ***
140 already downloaded, 0 remaining.

Stage 3a done.
status
ok    140
Name: count, dtype: int64


In [24]:
# --- Stage 3b: turn the cached content into diffs ---
# Local only, no network. Walks each skill's versions in publish order and
# diffs each one against the version before it.

import difflib

def read_content(skill_slug, version):
    path = content_path(skill_slug, version)
    if not path.exists():
        return None
    try:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    except (OSError, json.JSONDecodeError):
        return None

def pick_skill_md(files):
    """This version's main SKILL.md text — the substrate for topic modelling."""
    if not files:
        return None
    for name in files:
        if name.split("/")[-1].upper() == "SKILL.MD":
            return files[name]
    md_names = [n for n in files if n.lower().endswith(".md")]
    if md_names:
        return files[max(md_names, key=lambda n: len(files[n]))]
    return None

def bundle_diff(prev_files, curr_files):
    """Unified diff across every text file in either version, plus churn counts."""
    prev_files = prev_files or {}
    curr_files = curr_files or {}
    chunks = []
    added = removed = 0
    for name in sorted(set(prev_files) | set(curr_files)):
        before = (prev_files.get(name) or "").splitlines(keepends=True)
        after = (curr_files.get(name) or "").splitlines(keepends=True)
        if before == after:
            continue
        hunk = list(difflib.unified_diff(before, after,
                                         fromfile=f"a/{name}", tofile=f"b/{name}"))
        for line in hunk:
            if line.startswith(("+++", "---", "@@")):
                continue
            if line.startswith("+"):
                added += 1
            elif line.startswith("-"):
                removed += 1
        chunks.extend(hunk)
    return "".join(chunks), added, removed

diff_rows = []
n_missing = 0

ordered = versions_flat_df.sort_values(["skill_slug", "created_at"])
for skill_slug, group in ordered.groupby("skill_slug", sort=False):
    prev_files = None
    for row in group.itertuples():
        curr_files = read_content(skill_slug, row.version)
        if curr_files is None:
            # Not downloaded (or failed). Blank the predecessor too, so the next
            # version isn't diffed against the wrong ancestor and quietly lies.
            n_missing += 1
            prev_files = None
            continue
        if prev_files is None:
            # First version of the skill, or the one before it is missing —
            # no honest diff to compute.
            diff_text, added, removed = None, None, None
        else:
            diff_text, added, removed = bundle_diff(prev_files, curr_files)
        diff_rows.append({
            "skill_slug": skill_slug,
            "version": row.version,
            "diff_text": diff_text,
            "lines_added": added,
            "lines_removed": removed,
            "lines_changed": None if added is None else added + removed,
            "crawled_skill_content": pick_skill_md(curr_files),
        })
        prev_files = curr_files

diffs_df = pd.DataFrame(diff_rows, columns=[
    "skill_slug", "version", "diff_text",
    "lines_added", "lines_removed", "lines_changed", "crawled_skill_content",
])

n_with_diff = int(diffs_df["diff_text"].notna().sum())
print(f"{len(diffs_df)} versions with content, {n_with_diff} of them with a diff "
      f"against a predecessor. {n_missing} version(s) had no cached content.")

diffs_df.to_parquet("clawhub_diffs.parquet", index=False)
diffs_df[["skill_slug", "version", "lines_added", "lines_removed", "lines_changed"]].head(20)


304 versions with content, 222 of them with a diff against a predecessor. 177035 version(s) had no cached content.


,skill_slug,version,lines_added,lines_removed,lines_changed
0,84191879/agent-group,1.0.0,NaN,NaN,NaN
1,84191879/agent-group,1.0.1,2.0,2.0,4.0
2,84191879/agent-group,1.0.2,2365.0,133.0,2498.0
3,84191879/agent-group,1.0.3,92.0,2387.0,2479.0
4,aidankong/stm32-cubemx,1.0.0,NaN,NaN,NaN
5,aidankong/stm32-cubemx,1.0.1,2.0,2.0,4.0
6,aidankong/stm32-cubemx,1.0.2,190.0,190.0,380.0
7,aidankong/stm32-cubemx,1.0.3,52.0,2.0,54.0
8,arosstale/openclaw-memories,2.0.1,NaN,NaN,NaN
9,arosstale/openclaw-memories,2.0.2,124.0,814.0,938.0


In [25]:
# --- REHYDRATE df_all: reload the FROZEN snapshot after a kernel restart ---
# Deliberately does NOT re-resolve "main". The freeze happened once, on
# 2026-09-25, and the revision is recorded in dataset_provenance.json —
# re-running the original loader cell would pin a NEWER commit and quietly
# invalidate the row counts already reported.
import json
from pathlib import Path

from datasets import load_dataset

provenance = json.loads(Path("dataset_provenance.json").read_text())
frozen_revision = provenance["source_dataset_revision"]

dataset = load_dataset(
    provenance["source_dataset"],
    name="default",
    revision=frozen_revision,
    verification_mode="no_checks",
)
latest = dataset["latest"]
df_all = latest.to_pandas()

print(f"reloaded frozen snapshot @ {frozen_revision[:12]} "
      f"(frozen {provenance['source_dataset_frozen_at']})")
print(f"df_all: {len(df_all):,} rows x {len(df_all.columns)} columns")
assert len(df_all) == 77873, (
    f"expected 77,873 rows from the frozen revision, got {len(df_all):,} — "
    f"the pin in dataset_provenance.json may have been overwritten"
)

reloaded frozen snapshot @ 323d4b4bd899 (frozen 2026-09-25T09:01:35.555373+00:00)
df_all: 77,873 rows x 25 columns


In [26]:
# --- Final assembly: original dataset columns + everything crawled ---

vh = version_history_df.sort_values(["skill_slug", "created_at"]).copy()

# Where each version sits in its own skill's history, ordered by publish time
# (not semver — publish order is what "what changed in this release" means).
vh["version_index"] = vh.groupby("skill_slug").cumcount() + 1
vh["prev_version"] = vh.groupby("skill_slug")["version"].shift(1)
_prev_created = vh.groupby("skill_slug")["created_at"].shift(1)
vh["days_since_prev_version"] = (vh["created_at"] - _prev_created) / 86_400_000.0

# Gap between publishing and scanning. Seconds means this version was scanned at
# publish time under whatever pipeline was current then; months means it was
# re-scanned later under a newer one. Needed to judge whether two versions'
# ratings are on the same scale.
if "llm_checked_at" in vh.columns:
    vh["llm_scan_lag_days"] = (vh["llm_checked_at"] - vh["created_at"]) / 86_400_000.0

# Previous version's security outcome carried onto this row, so deltas can be
# computed later without touching the network again. llm_model comes along so a
# pair can be checked for scanner comparability:
#     comparable = (llm_model == prev_llm_model)
# or, more reliably, by whether both versions have a SkillSpector block.
_carry = [
    "crawled_skillspector_score",
    "crawled_skillspector_severity",
    "llm_normalized_status",
    "llm_model",
] + [f"llm_dim_{name}" for name in LLM_DIMENSIONS]

for col in _carry:
    if col in vh.columns:
        vh[f"prev_{col}"] = vh.groupby("skill_slug")[col].shift(1)

# Stage 3: diffs + per-version content
vh = vh.merge(diffs_df, on=["skill_slug", "version"], how="left")

# The original dataset's columns, attached ONLY to the row whose version matches
# the version HF snapshotted. Earlier-version rows keep these blank, which is
# correct — HF never recorded those versions.
final_df = vh.merge(
    df_all,
    left_on=["skill_slug", "version"],
    right_on=["skill_slug", "skill_version"],
    how="left",
)

n_rows = len(final_df)
n_skills = final_df["skill_slug"].nunique()
n_hf_matched = int(final_df["skill_version"].notna().sum())
print(f"{n_rows} rows across {n_skills} skills.")
print(f"{n_hf_matched} rows carry the original dataset's columns. "
      f"{n_skills - n_hf_matched} skill(s) matched nothing — most likely they published "
      f"a newer version after the HF snapshot was frozen, so its latest no longer "
      f"appears in their ClawHub history under that number.")

# Coverage of the candidate labels, so it's obvious which one is usable.
for col, name in [("llm_normalized_status", "LLM verdict"),
                  ("llm_dim_purpose_capability", "LLM dimension ratings"),
                  ("crawled_skillspector_score", "SkillSpector score")]:
    if col in final_df.columns:
        pct = final_df[col].notna().mean()
        print(f"  {name:<24} present on {pct:6.1%} of rows")

# Column order: crawled columns first, then the original dataset's, untouched.
crawled_cols = (
    [
        "skill_slug", "owner_handle", "slug", "version", "version_index",
        "created_at", "prev_version", "days_since_prev_version",
        "changelog_text", "changelog_source",
        "diff_text", "lines_added", "lines_removed", "lines_changed",
        "crawled_skill_content", "crawled_sha256",
        # security outcome — LLM scanner (broad coverage)
        "llm_status", "llm_verdict", "llm_normalized_status", "llm_confidence",
    ]
    + [f"llm_dim_{name}" for name in LLM_DIMENSIONS]
    + [
        "llm_dimensions_json", "llm_risk_summary_json",
        "llm_agentic_risk_findings_json",
        "llm_summary", "llm_guidance", "llm_findings",
        "llm_model", "llm_checked_at", "llm_scan_lag_days",
        # security outcome — SkillSpector (sparse)
        "crawled_skillspector_status", "crawled_skillspector_normalized_status",
        "crawled_skillspector_score", "crawled_skillspector_severity",
        "crawled_skillspector_recommendation", "crawled_skillspector_issue_count",
        "crawled_skillspector_checked_at",
        # VirusTotal — meaning depends on vt_source ("palm" = model-based with
        # prose analysis, "engines" = antivirus with analysis/verdict null)
        "vt_status", "vt_verdict", "vt_normalized_status", "vt_source",
        "vt_analysis", "vt_checked_at",
        # registry-level
        "crawled_security_status", "crawled_security_checked_at",
        "crawled_security_model", "crawled_has_scan_result",
        "crawled_has_warnings", "crawled_detail_status",
    ]
    # previous-version carries
    + [f"prev_{c}" for c in _carry]
)
hf_cols = [c for c in df_all.columns if c != "skill_slug"]
keep = [c for c in crawled_cols + hf_cols if c in final_df.columns]

# Guard: this list is hand-maintained, so a column added upstream would
# otherwise vanish here without a word. Say so instead.
dropped = [c for c in final_df.columns if c not in keep]
if dropped:
    print(f"\nWARNING: {len(dropped)} column(s) exist but are not in the keep "
          f"list, so they are being dropped: {dropped}")
    print("Add them to crawled_cols if they should survive.\n")

final_df = final_df[keep]

# Parquet, not CSV — diff_text and the prose columns hold embedded newlines and
# run to megabytes; CSV would be both huge and painful to read back.
final_df.to_parquet("clawhub_jit_dataset.parquet", index=False)

# A light CSV alongside it for eyeballing, with the bulky text columns dropped.
bulky = {"diff_text", "crawled_skill_content", "llm_dimensions_json",
         "llm_risk_summary_json", "llm_agentic_risk_findings_json",
         "llm_summary", "llm_guidance", "llm_findings", "vt_analysis",
         "skill_md_content", "skill_bundle_content",
         "clawscan_context", "clawscan_summary"}
preview_cols = [c for c in final_df.columns if c not in bulky]
final_df[preview_cols].to_csv("clawhub_jit_dataset_preview.csv", index=False)

print(f"\nSaved clawhub_jit_dataset.parquet — {len(final_df.columns)} columns:")
print(list(final_df.columns))
final_df[preview_cols].head(20)


177339 rows across 33492 skills.
33479 rows carry the original dataset's columns. 13 skill(s) matched nothing — most likely they published a newer version after the HF snapshot was frozen, so its latest no longer appears in their ClawHub history under that number.
  LLM verdict              present on  96.8% of rows
  LLM dimension ratings    present on  96.6% of rows
  SkillSpector score       present on  49.1% of rows

Saved clawhub_jit_dataset.parquet — 86 columns:
['skill_slug', 'owner_handle', 'slug', 'version', 'version_index', 'created_at', 'prev_version', 'days_since_prev_version', 'changelog_text', 'changelog_source', 'diff_text', 'lines_added', 'lines_removed', 'lines_changed', 'crawled_skill_content', 'crawled_sha256', 'llm_status', 'llm_verdict', 'llm_normalized_status', 'llm_confidence', 'llm_dim_purpose_capability', 'llm_dim_instruction_scope', 'llm_dim_install_mechanism', 'llm_dim_environment_proportionality', 'llm_dim_persistence_privilege', 'llm_dimensions_json', 'llm_

,skill_slug,owner_handle,slug,version,version_index,created_at,prev_version,days_since_prev_version,changelog_text,changelog_source,...,virustotal_suspicious_count,virustotal_harmless_count,virustotal_undetected_count,skillspector_status,skillspector_score,skillspector_severity,skillspector_issue_count,skillspector_issue_codes,skillspector_issue_categories,split
0,000wonderclaw/clawcv,000wonderclaw,clawcv,1.0.0,1,1774018324946,NaN,NaN,Initial release of ClawCV resume optimization ...,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000wonderclaw/clawcv,000wonderclaw,clawcv,1.0.2,2,1774326884515,1.0.0,3.571291,- Added English README file (README_EN.md) for...,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000wonderclaw/clawcv,000wonderclaw,clawcv,1.1.0,3,1775556109053,1.0.2,14.227136,ClawCV 1.1.0 introduces campus recruitment job...,user,...,0.0,0.0,64.0,suspicious,95.0,CRITICAL,22.0,"[AS1, AS1, RP1, RP1, RP1, RP1, RP1, RP1, SQP-1...","[Agent Snooping, MCP Rug Pull]",latest
3,00xmorty/conatus,00xmorty,conatus,1.0.0,1,1772063119292,NaN,NaN,The philosophical layer for AI agents. Spinoza...,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00xmorty/conatus,00xmorty,conatus,1.0.1,2,1772063689744,1.0.0,0.006602,Fix: republish to registry,user,...,0.0,0.0,66.0,suspicious,9.0,LOW,1.0,[SQP-2],[],latest
5,05u/jiuwu-agent,05u,jiuwu-agent,1.0.0,1,1774407899698,NaN,NaN,- Introduced jiuwu-agent skill for intelligent...,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,05u/jiuwu-agent,05u,jiuwu-agent,1.0.1,2,1775091906589,1.0.0,7.916746,- Added scripts/requirements.txt to manage Pyt...,user,...,0.0,0.0,66.0,suspicious,100.0,CRITICAL,15.0,"[LP3, PE3, PE3, PE3, PE3, PE3, PE3, SC1, SC4, ...","[MCP Least Privilege, Privilege Escalation, Su...",latest
7,05u/jiuwu-message,05u,jiuwu-message,1.0.0,1,1773882295554,NaN,NaN,- Initial release of jiuwu-message skill.\n- E...,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,05u/jiuwu-message,05u,jiuwu-message,1.0.1,2,1775091842974,1.0.0,13.999391,- Added initial scripts and requirements files...,user,...,0.0,0.0,66.0,suspicious,77.0,HIGH,6.0,"[LP3, PE3, PE3, SDI-2, SQP-1, SQP-2]","[MCP Least Privilege, Privilege Escalation]",latest
9,0731coderlee-sudo/wechat-publisher,0731coderlee-sudo,wechat-publisher,1.0.0,1,1770288770604,NaN,NaN,一键发布 Markdown 到微信公众号草稿箱,user,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# --- Coverage check: which security signal can actually serve as the label? ---
# No network needed — reads the Stage 2b checkpoint already on disk.

d = pd.read_json(VERSION_DETAIL_CHECKPOINT_PATH, lines=True, convert_dates=False)
print("call outcomes:")
print(d["crawled_detail_status"].value_counts(), "\n")

checks = versions_flat_df.merge(d, on=["skill_slug", "version"], how="left")
checks["is_newest"] = (
    checks["created_at"] == checks.groupby("skill_slug")["created_at"].transform("max")
)
checks["has_score"] = checks["crawled_skillspector_score"].notna()
checks["has_llm"] = checks["llm_normalized_status"].notna()
checks["has_dims"] = checks["llm_dim_purpose_capability"].notna()
checks["published"] = pd.to_datetime(checks["created_at"], unit="ms")

ok = checks[checks["crawled_detail_status"] == "ok"]
print(f"of {len(ok):,} successful calls:")
print(f"  LLM verdict present           {ok['has_llm'].mean():6.1%}")
print(f"  LLM dimension ratings present {ok['has_dims'].mean():6.1%}")
print(f"  SkillSpector score present    {ok['has_score'].mean():6.1%}\n")

print(pd.crosstab(checks["is_newest"], checks["has_score"],
                  rownames=["is newest version"], colnames=["has skillspector"]), "\n")

# When did SkillSpector enter ClawHub's pipeline? Look at NON-newest versions
# only — the newest of each skill gets re-scanned, which would mask the cutoff.
older = checks[~checks["is_newest"]].copy()
older["month"] = older["published"].dt.to_period("M")
by_month = older.groupby("month").agg(
    versions=("has_score", "size"),
    with_skillspector=("has_score", "sum"),
    pct_skillspector=("has_score", "mean"),
    pct_llm=("has_llm", "mean"),
)
print("coverage among non-newest versions, by publish month:")
print(by_month.to_string(formatters={
    "pct_skillspector": "{:.1%}".format,
    "pct_llm": "{:.1%}".format,
}))

# If there's a month where pct_skillspector jumps from ~0 to high, that's when
# SkillSpector went live — and skills whose versions all postdate it would have
# real per-version SkillSpector history, not just the LLM scanner.


call outcomes:
crawled_detail_status
ok                      176659
failed_after_retries       487
not_found                  193
Name: count, dtype: int64 

of 176,659 successful calls:
  LLM verdict present            97.2%
  LLM dimension ratings present  96.9%
  SkillSpector score present     49.3%

has skillspector   False  True 
is newest version              
False              89783  54064
True                 406  33086 

coverage among non-newest versions, by publish month:
         versions  with_skillspector pct_skillspector pct_llm
month                                                        
2026-01      1306                  6             0.5%    9.2%
2026-02     17595                278             1.6%   78.5%
2026-03     41584                471             1.1%   99.6%
2026-04     23125                899             3.9%   98.9%
2026-05     12176               4668            38.3%   99.9%
2026-06     11263              11103            98.6%   99.5%
2026-07     134

In [28]:
# --- What does the API actually return for an OLDER version? ---
# Dumps the raw security object, so we can see whether `scanners` is missing
# entirely, or present but empty / pending.

def dump_security(owner, slug, version, label=""):
    r = requests.get(
        f"https://clawhub.ai/api/v1/skills/{slug}/versions/{version}",
        params={"ownerHandle": owner},
        timeout=15,
    )
    print(f"--- {owner}/{slug} @ {version}  {label}  (HTTP {r.status_code}) ---")
    if r.status_code != 200:
        print(r.text[:300], "\n")
        return
    v = r.json().get("version") or {}
    print(json.dumps(v.get("security"), indent=2)[:1200], "\n")

# ponytail — the skill we tested by hand at the start. 4.9.0 is its newest.
dump_security("dietrichgebert", "ponytail", "4.7.0", "(older)")
dump_security("dietrichgebert", "ponytail", "4.9.0", "(newest)")

# A pair from your own crawl: same skill, an older version and its newest.
pair_slug = checks[(~checks["is_newest"]) & (~checks["has_score"])]["skill_slug"].iloc[0]
pair = checks[checks["skill_slug"] == pair_slug].sort_values("created_at")
older, newest_row = pair.iloc[0], pair.iloc[-1]
print(f"=== {pair_slug} ===")
dump_security(older["owner_handle"], older["slug"], older["version"], "(older)")
dump_security(newest_row["owner_handle"], newest_row["slug"], newest_row["version"], "(newest)")


--- dietrichgebert/ponytail @ 4.7.0  (older)  (HTTP 200) ---
{
  "status": "clean",
  "hasWarnings": true,
  "checkedAt": 1781655088646,
  "model": "gpt-5.5",
  "hasScanResult": true,
  "sha256hash": "b0c6e6456eb956ac1702eca185417dadb991cf339a72962d277725a9d1eabc69",
  "virustotalUrl": "https://www.virustotal.com/gui/file/b0c6e6456eb956ac1702eca185417dadb991cf339a72962d277725a9d1eabc69",
  "scanners": {
    "vt": {
      "status": "clean",
      "verdict": null,
      "normalizedStatus": "clean",
      "analysis": null,
      "source": "engines",
      "checkedAt": 1781612507702
    },
    "skillspector": {
      "status": "clean",
      "normalizedStatus": "clean",
      "score": 15,
      "severity": "LOW",
      "recommendation": "SAFE",
      "issueCount": 2,
      "checkedAt": 1781655065102
    },
    "llm": {
      "status": "clean",
      "verdict": "benign",
      "normalizedStatus": "clean",
      "confidence": "high",
      "summary": "Ponytail is a disclosed coding-style ins

In [29]:
# --- Do the VirusTotal / static columns carry ANY signal? ---
# Runs on the 80k rows already loaded — no network, instant. The point: a column
# that holds the same value on ~every row is not a feature. It can't help a
# model separate anything, so it isn't worth crawling per version even if the
# API would hand it over.

import numpy as np

def as_scalar(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return str(list(v))
    return v

check_cols = [
    "clawscan_verdict", "clawscan_confidence", "clawscan_model",
    "static_status", "static_finding_count", "static_reason_codes",
    "virustotal_status", "virustotal_malicious_count", "virustotal_suspicious_count",
    "virustotal_harmless_count", "virustotal_undetected_count",
    "skillspector_severity", "skillspector_score",
]

rows = []
for c in check_cols:
    if c not in df_all.columns:
        continue
    s = df_all[c].map(as_scalar)
    vc = s.value_counts(dropna=False)
    rows.append({
        "column": c,
        "n_unique": int(s.nunique(dropna=False)),
        "most_common": str(vc.index[0])[:24],
        "its_share": vc.iloc[0] / len(s),
        "2nd_most_common": str(vc.index[1])[:24] if len(vc) > 1 else "",
        "its_share_2": (vc.iloc[1] / len(s)) if len(vc) > 1 else 0.0,
    })

variance = pd.DataFrame(rows).sort_values("its_share", ascending=False)
pd.set_option("display.width", 160)
print(f"across all {len(df_all):,} rows:\n")
print(variance.to_string(index=False, formatters={
    "its_share": "{:.2%}".format, "its_share_2": "{:.2%}".format,
}))

print("\nA column whose top value covers ~100% of rows is constant — it cannot")
print("contribute to a model, so it is not worth a multi-hour per-version crawl.")


across all 77,873 rows:

                     column  n_unique most_common its_share          2nd_most_common its_share_2
           clawscan_verdict         4       clean    99.45%                malicious       0.37%
             clawscan_model         3        None    99.13%                  gpt-5.5       0.59%
        clawscan_confidence         4        high    93.38%                   medium       6.61%
       static_finding_count        46           0    92.35%                        1       4.91%
        static_reason_codes       190          []    92.35% ['suspicious.exposed_sec       1.80%
              static_status         4       clean    90.84%               suspicious       7.58%
virustotal_suspicious_count         2         0.0    90.53%                      nan       9.47%
  virustotal_harmless_count         2         0.0    90.53%                      nan       9.47%
 virustotal_malicious_count        15         0.0    90.31%                      nan       9.47%
     

In [30]:
# --- Is the rest of the API response hiding static_* or the VirusTotal counts? ---
# Our earlier dump only printed version.security. Print the whole shape this
# time, so nothing gets missed. Two versions: one newest (full pipeline) and
# one older (llm-only), to see what differs.

import numpy as np

def dump_shape(owner, slug, version, label=""):
    r = requests.get(
        f"https://clawhub.ai/api/v1/skills/{slug}/versions/{version}",
        params={"ownerHandle": owner},
        timeout=15,
    )
    print(f"=== {owner}/{slug} @ {version} {label} (HTTP {r.status_code}) ===")
    if r.status_code != 200:
        print(r.text[:300], "\n")
        return
    data = r.json()
    print("top level     :", list(data.keys()))
    v = data.get("version") or {}
    print("version       :", list(v.keys()))
    sec = v.get("security") or {}
    print("security      :", list(sec.keys()))
    scanners = sec.get("scanners") or {}
    print("scanners      :", list(scanners.keys()))
    vt = scanners.get("vt")
    print("vt            :", json.dumps(vt) if vt else "<null>")
    # 'static' is a scanner in the HF dataset — is it anywhere in this response?
    blob = json.dumps(data)
    for needle in ["static", "malicious", "harmless", "undetected", "reason_code", "reasonCode"]:
        print(f"  '{needle}' appears in response: {needle in blob}")
    print()

dump_shape("0range-x", "triple-layer-memory", "2.1.0", "(newest)")
dump_shape("0range-x", "triple-layer-memory", "2.0.1", "(older)")


=== 0range-x/triple-layer-memory @ 2.1.0 (newest) (HTTP 200) ===
top level     : ['skill', 'version']
version       : ['version', 'createdAt', 'changelog', 'changelogSource', 'license', 'files', 'security']
security      : ['status', 'hasWarnings', 'checkedAt', 'model', 'hasScanResult', 'sha256hash', 'virustotalUrl', 'scanners']
scanners      : ['vt', 'skillspector', 'llm']
vt            : {"status": "clean", "verdict": null, "normalizedStatus": "clean", "analysis": null, "source": "engines", "checkedAt": 1780089794789}
  'static' appears in response: False
  'malicious' appears in response: True
  'harmless' appears in response: False
  'undetected' appears in response: False
  'reason_code' appears in response: False
  'reasonCode' appears in response: False

=== 0range-x/triple-layer-memory @ 2.0.1 (older) (HTTP 200) ===
top level     : ['skill', 'version']
version       : ['version', 'createdAt', 'changelog', 'changelogSource', 'license', 'files', 'security']
security      : ['stat

In [31]:
# --- How many version pairs are actually comparable? ---
# Four things can disqualify a pair:
#   1. One side was never scanned (status pending / stale / error) - no label.
#   2. The two sides went through different scan PIPELINES. A version with a
#      SkillSpector block went through the newer one.
#   3. The two sides were scanned by different MODELS. Even inside the old
#      pipeline the model changed (gpt-5-mini early 2026, gpt-5.5 later).
#   4. (reported, not excluded) The scanners on one row are not contemporaneous
#      - see the timestamp block below.

import numpy as np

d = pd.read_json(VERSION_DETAIL_CHECKPOINT_PATH, lines=True, convert_dates=False)
m = (versions_flat_df
     .merge(d, on=["skill_slug", "version"], how="left")
     .sort_values(["skill_slug", "created_at"]))

# Scanned = some scanner returned something. crawled_has_scan_result alone is
# not enough: it arrives as a float with NaNs mixed in.
scanned = (m["llm_normalized_status"].notna() | m["crawled_skillspector_score"].notna())

m["pipeline"] = np.where(~scanned, "unscanned",
                np.where(m["crawled_skillspector_score"].notna(), "new", "old"))
# Strict key: pipeline AND model. New-pipeline rows report model as null, so
# they collapse into one profile, which is what we want.
m["scanner_profile"] = np.where(
    ~scanned, "unscanned",
    m["pipeline"] + "/" + m["llm_model"].fillna("none"),
)
m["scan_lag_days"] = (m["llm_checked_at"] - m["created_at"]) / 86_400_000

for col in ["pipeline", "scanner_profile", "crawled_sha256", "llm_normalized_status"]:
    m[f"prev_{col}"] = m.groupby("skill_slug")[col].shift(1)
m["has_prev"] = m.groupby("skill_slug").cumcount() > 0

print("versions by scanner profile:")
print(m["scanner_profile"].value_counts().to_string())
if "crawled_security_status" in m.columns:
    print("\nraw security status (pending / stale / error mean no usable scan):")
    print(m["crawled_security_status"].value_counts(dropna=False).to_string())
print()

pairs = m[m["has_prev"]]
both_scanned = (pairs["pipeline"] != "unscanned") & (pairs["prev_pipeline"] != "unscanned")
same_pipeline = pairs["pipeline"] == pairs["prev_pipeline"]
same_profile = pairs["scanner_profile"] == pairs["prev_scanner_profile"]

loose = both_scanned & same_pipeline      # same pipeline, model may differ
usable = both_scanned & same_profile      # same pipeline AND same model

print(f"{len(pairs):,} consecutive version pairs in total")
print(f"  {(~both_scanned).sum():,} ({(~both_scanned).mean():.1%}) have an unscanned side - no label")
print(f"  {(both_scanned & ~same_pipeline).sum():,} ({(both_scanned & ~same_pipeline).mean():.1%}) cross the pipeline change")
print(f"  {(loose & ~same_profile).sum():,} ({(loose & ~same_profile).mean():.1%}) same pipeline but a DIFFERENT MODEL")
print(f"\n  {loose.sum():,} ({loose.mean():.1%}) usable under the loose rule (same pipeline)")
print(f"  {usable.sum():,} ({usable.mean():.1%}) USABLE under the strict rule (same pipeline + model)\n")

print(pd.crosstab(pairs["prev_scanner_profile"], pairs["scanner_profile"],
                  rownames=["previous version"], colnames=["this version"]), "\n")

print("scan lag in days (publish -> llm scan), scanned versions only:")
print(m.loc[scanned, "scan_lag_days"].describe().to_string(), "\n")

# --- Are the three scanners contemporaneous? ------------------------------
# No. The probe showed a version whose LLM and SkillSpector results are dated
# 14 Sep 2026 while its VirusTotal result is dated 29 May 2026 - 108 days
# earlier. A re-scan runs the LLM and SkillSpector fresh but carries the old
# VirusTotal record forward, so vt_* and llm_* on one row can describe scans
# months apart. That also explains why vt_source does not track the pipeline
# split: a "palm"-mode VT record can sit on a new-pipeline row because it was
# inherited rather than re-run.
#
# Upshot: never read crawled_security_checked_at as "when this version was
# scanned" - it is the max across scanners. Use the per-scanner timestamp that
# matches the field you are looking at.
if {"vt_checked_at", "llm_checked_at"} <= set(m.columns):
    both_ts = m["vt_checked_at"].notna() & m["llm_checked_at"].notna()
    if both_ts.any():
        gap = (m.loc[both_ts, "llm_checked_at"] - m.loc[both_ts, "vt_checked_at"]) / 86_400_000
        stale = int((gap > 7).sum())
        print(f"scanner time spread ({int(both_ts.sum()):,} versions with both timestamps):")
        print(f"  llm minus vt, in days: median {gap.median():.1f}, "
              f"90th pct {gap.quantile(0.9):.1f}, max {gap.max():.1f}")
        print(f"  {stale:,} ({stale / int(both_ts.sum()):.1%}) carry a VirusTotal result "
              f"more than a week older than the LLM result\n")

# --- Noise floor ----------------------------------------------------------
# Some releases are content-identical republishes. Where the bundle hash is
# unchanged but the rating moved anyway, that movement is pure scanner
# variance, not an effect of the change.
identical = pairs["crawled_sha256"].eq(pairs["prev_crawled_sha256"]) & pairs["crawled_sha256"].notna()
rating_moved = pairs["llm_normalized_status"].ne(pairs["prev_llm_normalized_status"])
noise_base = identical & usable
if noise_base.sum():
    flip = rating_moved[noise_base].mean()
    print(f"noise floor: {int(noise_base.sum()):,} usable pairs have IDENTICAL content "
          f"(same sha256) on both sides.")
    print(f"  of those, {int(rating_moved[noise_base].sum()):,} ({flip:.1%}) still changed "
          f"their LLM verdict.")
    print("  That share is the label's own instability - a lower bound on how much")
    print("  of any measured effect is scanner noise rather than the change itself.")
else:
    print("noise floor: no usable pairs with identical content, so it can't be estimated.")

# Cleanest cohort: every version scanned, all under one profile.
profiles = m.groupby("skill_slug")["scanner_profile"].agg(set)
single = profiles[profiles.apply(lambda s: len(s) == 1 and "unscanned" not in s)].index
print(f"\n{len(single):,} of {m['skill_slug'].nunique():,} skills have every version "
      f"scanned under one profile:")
print(m[m["skill_slug"].isin(single)].groupby("scanner_profile")["skill_slug"].nunique().to_string())


versions by scanner profile:
scanner_profile
old/gpt-5-mini               75536
new/none                     62573
new/gpt-5.5                  24577
old/gpt-5.5                   8846
unscanned                     5693
old/codex-security-worker       61
old/none                        47
old/gpt-4.1-mini                 6

raw security status (pending / stale / error mean no usable scan):
crawled_security_status
suspicious    86074
clean         85180
None           2676
pending        2337
NaN             680
error           344
malicious        48

143,847 consecutive version pairs in total
  5,588 (3.9%) have an unscanned side - no label
  22,050 (15.3%) cross the pipeline change
  9,865 (6.9%) same pipeline but a DIFFERENT MODEL

  116,209 (80.8%) usable under the loose rule (same pipeline)
  106,344 (73.9%) USABLE under the strict rule (same pipeline + model)

this version               new/gpt-5.5  new/none  old/codex-security-worker  old/gpt-4.1-mini  old/gpt-5-mini  old/gpt-5.

In [32]:
# --- Dataset characterisation: two summary tables ---
# Run AFTER the comparability cell above - it reuses m / pairs / both_scanned /
# same_pipeline / same_profile from there.
#
# Table A is the equivalent of Table 3 in the ClawHub paper, but with a column
# their corpus did not need: scanner generation. Pooling old and new scans into
# one coverage figure would average across different measurement regimes and
# hide the thing that most constrains this dataset.
#
# Note the three states, following the paper's own distinction between "rows
# with source" (65,873) and "resolved status" (65,640, with 233 stale):
#   no record  - the scanner object is absent/null for this version
#   unresolved - a record exists but carries no usable status
#   resolved   - a usable status, which is then clean or positive
# "Positive" follows the paper: status is suspicious or malicious. clean,
# stale, error and missing are all non-positive.

POSITIVE = {"suspicious", "malicious"}

SCANNERS = {
    "LLM (ClawScan)": "llm_normalized_status",
    "SkillSpector":   "crawled_skillspector_normalized_status",
    "VirusTotal":     "vt_normalized_status",
}

rows = []
for gen, grp in m.groupby("pipeline", dropna=False):
    for name, col in SCANNERS.items():
        if col not in grp.columns:
            continue
        n = len(grp)
        resolved = grp[col].notna()
        positive = grp[col].isin(POSITIVE)
        rows.append({
            "generation": gen,
            "scanner": name,
            "versions": n,
            "resolved": int(resolved.sum()),
            "resolved_pct": resolved.mean(),
            "positive": int(positive.sum()),
            "positive_of_all": positive.mean(),
            "positive_of_resolved": (positive.sum() / resolved.sum()) if resolved.sum() else float("nan"),
        })

coverage = pd.DataFrame(rows).sort_values(["scanner", "generation"])
pd.set_option("display.width", 200)
print("TABLE A - scanner coverage and positive rate, by scanner generation")
print(f"(positive = status is suspicious or malicious)\n")
print(coverage.to_string(index=False, formatters={
    "resolved_pct": "{:.1%}".format,
    "positive_of_all": "{:.1%}".format,
    "positive_of_resolved": "{:.1%}".format,
}))

# Same thing pooled, for comparison with the paper's single-regime numbers.
pooled = []
for name, col in SCANNERS.items():
    if col not in m.columns:
        continue
    resolved = m[col].notna()
    positive = m[col].isin(POSITIVE)
    pooled.append({
        "scanner": name, "versions": len(m),
        "resolved": int(resolved.sum()), "resolved_pct": resolved.mean(),
        "positive": int(positive.sum()), "positive_of_all": positive.mean(),
    })
print("\npooled across generations (comparable in shape to the paper's Table 3,")
print("but remember it blends measurement regimes):")
print(pd.DataFrame(pooled).to_string(index=False, formatters={
    "resolved_pct": "{:.1%}".format, "positive_of_all": "{:.1%}".format}))

# --- Table B: what survives, step by step --------------------------------
total = len(pairs)
lost_unscanned = int((~both_scanned).sum())
lost_pipeline = int((both_scanned & ~same_pipeline).sum())
lost_model = int((both_scanned & same_pipeline & ~same_profile).sum())
surviving = int((both_scanned & same_profile).sum())

funnel = pd.DataFrame([
    {"step": "all consecutive version pairs", "lost": 0, "remaining": total},
    {"step": "- one side never scanned", "lost": lost_unscanned,
     "remaining": total - lost_unscanned},
    {"step": "- crosses the pipeline change", "lost": lost_pipeline,
     "remaining": total - lost_unscanned - lost_pipeline},
    {"step": "- scanned by different models", "lost": lost_model,
     "remaining": surviving},
])
funnel["remaining_pct"] = funnel["remaining"] / total if total else float("nan")

print("\n\nTABLE B - labelled pairs surviving each exclusion")
print("(a pair is one version compared against the version before it)\n")
print(funnel.to_string(index=False, formatters={"remaining_pct": "{:.1%}".format}))

print(f"\n{surviving:,} usable pairs across "
      f"{pairs.loc[both_scanned & same_profile, 'skill_slug'].nunique():,} skills.")
print("That is the number that characterises this dataset - the equivalent of")
print("the paper's 67,453 rows.")


TABLE A - scanner coverage and positive rate, by scanner generation
(positive = status is suspicious or malicious)

generation        scanner  versions  resolved resolved_pct  positive positive_of_all positive_of_resolved
       new LLM (ClawScan)     87150     87150       100.0%     43805           50.3%                50.3%
       old LLM (ClawScan)     84496     84496       100.0%     42317           50.1%                50.1%
 unscanned LLM (ClawScan)      5693         0         0.0%         0            0.0%                  NaN
       new   SkillSpector     87150     87150       100.0%     62232           71.4%                71.4%
       old   SkillSpector     84496        83         0.1%         0            0.0%                 0.0%
 unscanned   SkillSpector      5693         0         0.0%         0            0.0%                  NaN
       new     VirusTotal     87150     64793        74.3%      2290            2.6%                 3.5%
       old     VirusTotal     84496 

In [33]:
# --- Record the crawl window alongside the dataset freeze ---
# A live registry cannot be snapshotted atomically: Stage 1, 2a, 2b and 3a run
# over hours or days, so the dataset is a smear across a window rather than a
# single instant. That is normal for mining studies and is handled by REPORTING
# the window, not by chasing a purity that re-running cannot buy either.
#
# This appends per-stage timing and row counts to dataset_provenance.json, so
# the methods section can state exactly when each layer was collected.

from datetime import datetime, timezone

CHECKPOINTS = {
    "stage1_version_counts": Path("clawhub_version_counts.jsonl"),
    "stage2a_version_lists": Path("clawhub_version_lists.jsonl"),
    "stage2b_security":      Path("clawhub_version_security.jsonl"),
    "stage3a_content_index": Path("clawhub_content_index.jsonl"),
}

def describe(p):
    if not p.exists():
        return {"exists": False}
    with p.open() as f:
        rows = sum(1 for line in f if line.strip())
    st = p.stat()
    return {
        "exists": True,
        "rows": rows,
        "size_mb": round(st.st_size / 1_048_576, 1),
        "last_written_utc": datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat(),
    }

crawl = {name: describe(p) for name, p in CHECKPOINTS.items()}

try:
    with open("dataset_provenance.json") as f:
        provenance = json.load(f)
except FileNotFoundError:
    provenance = {}

provenance["crawl"] = crawl
provenance["crawl_recorded_utc"] = datetime.now(timezone.utc).isoformat()
with open("dataset_provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print(json.dumps(provenance, indent=2))
print("\nNote: last_written_utc is when each crawl FINISHED, not when it started.")
print("If you want the start too, note it by hand when you kick a stage off.")


{
  "source_dataset": "OpenClaw/clawhub-security-signals-live",
  "source_dataset_revision": "323d4b4bd899884e6726b97d74158654281960e4",
  "source_dataset_frozen_at": "2026-09-25T09:01:35.555373+00:00",
  "crawl": {
    "stage1_version_counts": {
      "exists": true,
      "rows": 77873,
      "size_mb": 5.9,
      "last_written_utc": "2026-09-24T13:47:18.026682+00:00"
    },
    "stage2a_version_lists": {
      "exists": true,
      "rows": 33527,
      "size_mb": 68.2,
      "last_written_utc": "2026-09-25T06:54:44.257906+00:00"
    },
    "stage2b_security": {
      "exists": true,
      "rows": 177339,
      "size_mb": 829.0,
      "last_written_utc": "2026-09-29T09:42:10.640973+00:00"
    },
    "stage3a_content_index": {
      "exists": true,
      "rows": 140,
      "size_mb": 0.0,
      "last_written_utc": "2026-09-29T09:48:16.711382+00:00"
    }
  },
  "crawl_recorded_utc": "2026-10-01T12:20:35.279224+00:00"
}

Note: last_written_utc is when each crawl FINISHED, not when it s

In [34]:
import requests
import json

r = requests.get("https://clawhub.ai/api/v1/skills/ponytail/scan",
                 params={"ownerHandle": "dietrichgebert", "version": "4.9.0"}, timeout=20)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:3000] if r.status_code == 200 else r.text[:300])

200
{
  "skill": {
    "slug": "ponytail",
    "displayName": "ponytail"
  },
  "version": {
    "version": "4.9.0",
    "createdAt": 1786137349071,
    "changelogSource": "auto"
  },
  "moderation": null,
  "security": {
    "status": "clean",
    "hasWarnings": true,
    "checkedAt": 1789550601287,
    "model": null,
    "hasScanResult": true,
    "sha256hash": "12f2bb85c1329536e8846d10c2905233aa968e1f25a4ce14069c8f6deae0fcb3",
    "virustotalUrl": "https://www.virustotal.com/gui/file/12f2bb85c1329536e8846d10c2905233aa968e1f25a4ce14069c8f6deae0fcb3",
    "scanners": {
      "vt": {
        "status": "clean",
        "verdict": null,
        "normalizedStatus": "clean",
        "analysis": null,
        "source": "engines",
        "checkedAt": 1786137709432
      },
      "skillspector": {
        "status": "suspicious",
        "normalizedStatus": "suspicious",
        "score": 13,
        "severity": "LOW",
        "recommendation": "CAUTION",
        "issueCount": 2,
        "chec

In [35]:
# --- What does the OpenAPI spec say the scanners actually are? ---
import json, re

spec = requests.get("https://clawhub.ai/api/v1/openapi.json", timeout=30).json()
print("OpenAPI", spec.get("openapi"), "-", (spec.get("info") or {}).get("title"),
      (spec.get("info") or {}).get("version"))

TARGETS = {"scanners", "llm", "vt", "virustotal", "skillspector",
           "static", "staticscan", "clawscan"}

def walk(node, path=""):
    """Find every place the spec mentions a scanner, and show its schema."""
    if isinstance(node, dict):
        for key, val in node.items():
            here = f"{path}.{key}" if path else key
            if key.lower() in TARGETS:
                print("\n" + "=" * 70)
                print(here)
                print("=" * 70)
                print(json.dumps(val, indent=2)[:2500])
            walk(val, here)
    elif isinstance(node, list):
        for i, val in enumerate(node):
            walk(val, f"{path}[{i}]")

walk(spec)

# Any description text mentioning a scanner by name, wherever it sits
print("\n" + "=" * 70)
print("descriptions mentioning a scanner")
print("=" * 70)
for m in re.finditer(r'"description"\s*:\s*"([^"]{0,400})"', json.dumps(spec)):
    d = m.group(1)
    if re.search(r"clawscan|skillspector|virustotal|static analys|llm", d, re.I):
        print(" -", d)

with open("clawhub_openapi.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, indent=2)
print("\n[saved to clawhub_openapi.json]")

OpenAPI 3.1.0 - ClawHub API 1.0.0

components.schemas.PackageVersionResponse.properties.version.properties.staticScan
{
  "type": [
    "object",
    "null"
  ],
  "additionalProperties": true
}

components.schemas.SecuritySnapshot.properties.scanners
{
  "type": "object",
  "additionalProperties": true
}

descriptions mentioning a scanner

[saved to clawhub_openapi.json]


In [36]:
# --- Is scanners.llm the ClawScan verdict, or a separate LLM scanner? ---
# The OpenAPI spec is silent: SecuritySnapshot.scanners is untyped
# ({"type":"object","additionalProperties":true}). So test it against data.
#
# The HF snapshot recorded clawscan_* at each skill's LATEST version; the crawl
# recorded llm_* at every version including that one. Where they overlap:
#   near-perfect agreement -> scanners.llm IS the ClawScan verdict
#   real divergence        -> scanners.llm is its own scanner, ClawScan above it
#
# Reads the parquet from disk, so it needs nothing else in memory.

import pandas as pd

final_df = pd.read_parquet("clawhub_jit_dataset.parquet")
m = final_df[final_df["skill_version"].notna()].copy()
print(f"{len(final_df):,} rows total; {len(m):,} carry both HF clawscan_* "
      f"and crawled llm_*\n")

PAIRS = [("llm_verdict",    "clawscan_verdict"),
         ("llm_confidence", "clawscan_confidence"),
         ("llm_model",      "clawscan_model"),
         ("llm_summary",    "clawscan_summary")]

def norm(s):
    return s.astype(str).str.strip().str.lower()

for crawled, hf in PAIRS:
    if crawled not in m.columns or hf not in m.columns:
        print(f"{crawled:<16} vs {hf:<22} column missing"); continue
    both = m[m[crawled].notna() & m[hf].notna()]
    if both.empty:
        print(f"{crawled:<16} vs {hf:<22} no rows with both present"); continue
    agree = (norm(both[crawled]) == norm(both[hf])).mean()
    print(f"{crawled:<16} vs {hf:<22} identical on {agree:6.1%} of {len(both):,} rows")

print("\nverdict crosstab (rows = crawled llm_verdict, cols = HF clawscan_verdict):")
print(pd.crosstab(m["llm_verdict"], m["clawscan_verdict"], dropna=False))

print("\nmodel crosstab — both being null in the same places is itself evidence:")
print(pd.crosstab(m["llm_model"].fillna("<null>"),
                  m["clawscan_model"].fillna("<null>"), dropna=False))

print("\ncontrol: SkillSpector should agree, since both sides name it explicitly")
ss = m[m["crawled_skillspector_score"].notna() & m["skillspector_score"].notna()]
if len(ss):
    print(f"  score identical on "
          f"{(ss['crawled_skillspector_score'] == ss['skillspector_score']).mean():.1%} "
          f"of {len(ss):,} rows")
else:
    print("  no overlapping rows — check the merge")

177,339 rows total; 33,479 carry both HF clawscan_* and crawled llm_*

llm_verdict      vs clawscan_verdict       identical on   0.0% of 33,308 rows
llm_confidence   vs clawscan_confidence    identical on 100.0% of 33,308 rows
llm_model        vs clawscan_model         identical on 100.0% of 136 rows
llm_summary      vs clawscan_summary       identical on  99.0% of 33,308 rows

verdict crosstab (rows = crawled llm_verdict, cols = HF clawscan_verdict):
clawscan_verdict  clean  suspicious  unknown
llm_verdict                                 
benign            12313           0        1
suspicious        20984          10        0
NaN                 171           0        0

model crosstab — both being null in the same places is itself evidence:
clawscan_model  <null>  gpt-5-mini  gpt-5.5
llm_model                                  
<null>           33342           1        0
gpt-5-mini           0          95        0
gpt-5.5              0           0       41

control: SkillSpector sho

In [37]:
print("which crawled field corresponds to the HF clawscan_verdict?\n")
for col in ["llm_normalized_status", "llm_status", "crawled_security_status"]:
    if col not in m.columns:
        continue
    both = m[m[col].notna() & m["clawscan_verdict"].notna()]
    agree = (norm(both[col]) == norm(both["clawscan_verdict"])).mean()
    print(f"  {col:<26} identical on {agree:6.1%} of {len(both):,} rows")

print("\nllm_normalized_status vs clawscan_verdict:")
print(pd.crosstab(m["llm_normalized_status"], m["clawscan_verdict"], dropna=False))

print("\na few of the 20,984 'model says suspicious, registry says clean' rows:")
gap = m[(m["llm_verdict"] == "suspicious") & (m["clawscan_verdict"] == "clean")]
for _, r in gap.head(3).iterrows():
    print(f"\n  {r['skill_slug']} @ {r['version']}  conf={r['llm_confidence']}")
    print(f"    {str(r['llm_summary'])[:300]}")

which crawled field corresponds to the HF clawscan_verdict?

  llm_normalized_status      identical on  37.0% of 33,308 rows
  llm_status                 identical on  37.0% of 33,308 rows
  crawled_security_status    identical on  37.0% of 33,308 rows

llm_normalized_status vs clawscan_verdict:
clawscan_verdict       clean  suspicious  unknown
llm_normalized_status                            
clean                  12313           0        1
suspicious             20984          10        0
NaN                      171           0        0

a few of the 20,984 'model says suspicious, registry says clean' rows:

  000wonderclaw/clawcv @ 1.1.0  conf=high
    ClawCV appears to be a real resume helper, but it needs review because it runs an unpinned npm MCP server, sends sensitive resume data to WonderCV, and exposes API keys through setup examples.

  00xmorty/conatus @ 1.0.1  conf=high
    This skill is not malicious, but it should be reviewed because it tells an agent to repeatedly rea

In [38]:
import pandas as pd, json
from collections import Counter

final_df = pd.read_parquet("clawhub_jit_dataset.parquet")
m = final_df[final_df["skill_version"].notna()]
gap = m[(m["llm_normalized_status"] == "suspicious") & (m["clawscan_verdict"] == "clean")]

ctx = gap["clawscan_context"].dropna()

top = Counter()
sub = {}
for d in ctx.head(3000):
    if not isinstance(d, dict):
        continue
    for k, v in d.items():
        top[k] += 1
        if isinstance(v, dict):
            sub.setdefault(k, Counter()).update(v.keys())

print("top-level keys of clawscan_context:")
for k, n in top.most_common():
    print(f"  {k:<22} on {n:,} rows")
for k, c in sub.items():
    print(f"\n  {k} sub-keys: {sorted(c)}")

# any field called verdict/status anywhere in the structure
def find(d, want=("verdict", "status", "normalized"), path=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            p = f"{path}.{k}" if path else k
            if any(w in k.lower() for w in want) and not isinstance(v, (dict, list)):
                out[p] = v
            out.update(find(v, want, p))
    return out

print("\nverdict/status fields on 3 gap rows (clawscan_verdict says 'clean'):")
for d in ctx.head(3):
    print(" ", find(d))

# does a context field reproduce the crawled llm status?
rows = []
for i, d in zip(gap.index[:3000], ctx.head(3000)):
    f = find(d)
    f["_clawscan_verdict"] = gap.at[i, "clawscan_verdict"]
    f["_llm_normalized"] = gap.at[i, "llm_normalized_status"]
    rows.append(f)
ctx_df = pd.DataFrame(rows)
print("\ncontext fields vs the two verdicts:")
for c in [c for c in ctx_df.columns if not c.startswith("_")]:
    if ctx_df[c].nunique(dropna=True) <= 8:
        print(f"\n{c}:")
        print(ctx_df[c].value_counts(dropna=False).to_string())

top-level keys of clawscan_context:
  skillspector           on 3,000 rows
  static                 on 3,000 rows
  virustotal             on 3,000 rows

  skillspector sub-keys: ['checked_at', 'issue_codes', 'issue_count', 'issues', 'scanner_version', 'score', 'severity', 'status', 'verdict']

  static sub-keys: ['checked_at', 'finding_count', 'reason_codes', 'scanner_version', 'status', 'summary', 'verdict']

  virustotal sub-keys: ['checked_at', 'engine_stats', 'scanner_version', 'status', 'verdict']

verdict/status fields on 3 gap rows (clawscan_verdict says 'clean'):
  {'skillspector.status': 'suspicious', 'skillspector.verdict': 'DO_NOT_INSTALL', 'static.status': 'clean', 'static.verdict': 'clean', 'virustotal.status': 'clean', 'virustotal.verdict': None}
  {'skillspector.status': 'suspicious', 'skillspector.verdict': 'CAUTION', 'static.status': 'clean', 'static.verdict': 'clean', 'virustotal.status': 'clean', 'virustotal.verdict': 'benign'}
  {'skillspector.status': 'suspicious'

In [11]:
import pandas as pd

m = final_df[final_df["skill_version"].notna()].copy()
for scanner in ["skillspector", "static", "virustotal"]:
    m[f"{scanner}_ctx"] = m["clawscan_context"].map(
        lambda d, s=scanner: (d or {}).get(s, {}).get("status")
        if isinstance(d, dict) else None)

print("published verdict vs each scanner's own status:\n")
for scanner in ["skillspector", "static", "virustotal"]:
    print(f"--- {scanner} ---")
    print(pd.crosstab(m[f"{scanner}_ctx"], m["clawscan_verdict"], dropna=False), "\n")

print("what actually drives a NON-clean published verdict:")
nonclean = m[m["clawscan_verdict"].isin(["suspicious", "malicious"])]
print(f"  {len(nonclean):,} rows")
for scanner in ["skillspector", "static", "virustotal"]:
    pos = nonclean[f"{scanner}_ctx"].isin(["suspicious", "malicious"]).mean()
    print(f"  {scanner:<14} positive on {pos:6.1%} of them")

AttributeError: 'NoneType' object has no attribute 'get'

In [39]:
import requests

counts_df = pd.read_json("clawhub_version_counts.jsonl", lines=True, convert_dates=False)
big = counts_df[counts_df["status"] == "ok"].nlargest(1, "versions").iloc[0]
owner, _, slug = big["skill_slug"].partition("/")
print(f"{big['skill_slug']} — Stage 1 counted {int(big['versions'])} versions")

for lim in [200, 500]:
    r = requests.get(f"https://clawhub.ai/api/v1/skills/{slug}/versions",
                     params={"ownerHandle": owner, "limit": lim}, timeout=30)
    d = r.json() if r.status_code == 200 else {}
    print(f"  limit={lim:<4} HTTP {r.status_code}  items={len(d.get('items', [])):<4} "
          f"nextCursor={'set' if d.get('nextCursor') else 'null'}")

onlyloveher/shield-cn — Stage 1 counted 1294 versions
  limit=200  HTTP 200  items=50   nextCursor=set
  limit=500  HTTP 200  items=50   nextCursor=set
